# Détection automatique de faux scans et d'anomalies dans les bases d'imagerie médicale
### Approche hybride apprentissage profond + forensique numérique — carnet d'expérimentation (LIDC-IDRI)

**Auteur :** Ing. Mendoua Vinny Jordan — Master Recherche SDIA, ENSPD, 2025-2026

Ce carnet implémente le **protocole complet** décrit au chapitre 2 du mémoire :

| § | Contenu |
|---|---|
| 0 | Installation, configuration, graines, GPU |
| 1 | Lecture de LIDC-IDRI (pydicom / pylidc), conversion en HU, masque pulmonaire |
| 2 | Partition 70/15/15 **par patient**, inventaire des coupes |
| 3 | Attaques classiques : copier-déplacer, inpainting, bruit, FGSM (+ camouflage du bruit) |
| 4 | GAN d'inpainting conditionnel (pix2pix) : injection / suppression de nodules |
| 5 | DDPM + inpainting de type RePaint |
| 5 bis | Construction du jeu falsifié (coupes + masques de vérité terrain) |
| 6 | Branche forensique (résidus SRM/KV, autocorrélation, spectre, incohérence locale, duplication) |
| 7 | Branche CNN : couche contrainte de Bayar + SRM → EfficientNet-B0 → U-Net, perte BCE + Dice |
| 8 | Branche diffusion : score d'erreur de reconstruction |
| 9 | Fusion local → global, évaluation (test interne, CT-GAN externe, robustesse) et **génération du tableau 3.6** |
| 10 | Module de sécurité hospitalier : SHA-256, journal d'audit chaîné, politique d'alerte, interface Gradio |
| 11 | Scripts de la preuve de concept SimCT-Tamper (reproduction des chiffres du chapitre 3) |
| 12 | Figures calculées (résidus, bruit, copier-déplacer), sur une coupe réelle LIDC-IDRI si disponible |

**Matériel conseillé :** GPU ≥ 8 Go : Kaggle / Colab (T4, P100) ou GPU local, y compris AMD Radeon RX 7600 sous Windows avec PyTorch ROCm. Sur Kaggle : activer **GPU** et **Internet**, puis « Save & Run All ». Les scanners LIDC-IDRI retenus sont **téléchargés automatiquement depuis le TCIA**, lus puis supprimés au fur et à mesure (l'espace disque reste sous quelques Go). L'ensemble tourne en 3 à 5 heures avec 300 patients.

> Règle forensique fondamentale : **ne jamais rééchantillonner ni exporter en 8 bits** les images avant analyse (cf. §3.1.4 du mémoire). Toutes les opérations travaillent sur les HU natifs 16 bits.

## 0. Installation et configuration

In [ ]:
# Installation automatique des paquets manquants (Kaggle / Colab / local)
import importlib.util, subprocess, sys
_need = {"pydicom": "pydicom", "pylidc": "pylidc", "SimpleITK": "SimpleITK", "timm": "timm", "skimage": "scikit-image",
         "cv2": "opencv-python-headless", "tabulate": "tabulate", "gdown": "gdown"}
_missing = [pkg for mod, pkg in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("installation :", _missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=False)

In [ ]:
import os, json, math, time, random, hashlib, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from scipy.stats import kurtosis
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")

# Racine des données : /kaggle/working sur Kaggle, /content sur Colab, Documents\GPU_ROCm sur un PC Windows local
if Path("/kaggle/working").exists():   BASE = Path("/kaggle/working")
elif Path("/content").exists():        BASE = Path("/content")
else:                                  BASE = Path.home() / "Documents" / "GPU_ROCm"
CFG = dict(
    LIDC_ROOT=str((Path("/tmp") if Path("/kaggle/working").exists() else BASE) / "LIDC-IDRI"),   # DICOM LIDC-IDRI (téléchargés automatiquement depuis le TCIA si absents)
    CTGAN_ROOT=str((Path("/tmp") if Path("/kaggle/working").exists() else BASE) / "CT-GAN"),     # jeu UCI id=520 (test externe), téléchargé si possible
    OUT=str(BASE / "out"),
    FAST=True,                           # sous-ensemble de patients
    MAX_PATIENTS=300,                    # utilisé si FAST (300 patients : 210 entraînement, 45 validation, 45 test)
    DOWNLOAD_LIDC=True,                  # télécharger depuis le TCIA les scanners absents
    KEEP_DICOM=False,                    # supprimer chaque scanner après lecture (économie de disque)
    DOWNLOAD_CTGAN=True,                 # tenter de télécharger le jeu CT-GAN (6 Go) pour le test externe
    N_JOBS=max(1, (os.cpu_count() or 2)),
    SLICES_PER_PATIENT=12,
    P_AUTH=0.5,                          # proportion de coupes authentiques
    PATCH=64, STRIDE=32,
    SEED=42,
    GAN_EPOCHS=30, DDPM_EPOCHS=40, CNN_EPOCHS=30, NODCLS_EPOCHS=3,
    BATCH=128, LR=3e-4,
    T_DIFF=1000, T_STAR=150,
)
OUT = Path(CFG["OUT"]); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "slices").mkdir(exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if os.name == "nt" else 2   # sous Windows, les workers DataLoader ne voient pas les classes du notebook
T_START = time.time()
def chrono(msg): print(f"[{(time.time() - T_START) / 60:6.1f} min] {msg}", flush=True)

def pmap(fn, items, chunksize=4):
    # Parallélisation CPU (fork sous Linux) ; repli séquentiel sous Windows ou en cas d'erreur.
    items = list(items)
    if os.name != "nt" and CFG["N_JOBS"] > 1 and len(items) > 8:
        try:
            import multiprocessing as mp
            with mp.get_context("fork").Pool(CFG["N_JOBS"]) as pool:
                return pool.map_async(fn, items, chunksize=chunksize).get(timeout=4 * 3600)
        except Exception as e:
            print("parallélisation indisponible, passage séquentiel :", e)
    return [fn(it) for it in items]
print("Périphérique :", DEVICE, "| PyTorch", torch.__version__,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

# Radeon sous Windows (ROCm) : la BatchNorm de MIOpen doit être compilée à la volée et cette
# compilation échoue (« type_traits file not found »). On force la BatchNorm native de PyTorch
# et on garde MIOpen pour les convolutions. Le premier passage est lent (recherche des noyaux
# MIOpen, quelques minutes), les suivants utilisent le cache.
if DEVICE.type == "cuda" and getattr(torch.version, "hip", None) and os.name == "nt":
    _bn_miopen = F.batch_norm
    def _bn_native(*args, **kwargs):
        with torch.backends.cudnn.flags(enabled=False):
            return _bn_miopen(*args, **kwargs)
    F.batch_norm = _bn_native
    print("Correctif ROCm Windows actif : BatchNorm native, convolutions MIOpen.")

## 1. Lecture de LIDC-IDRI et prétraitement

Les volumes sont lus avec **pylidc**, qui gère l'ordre des coupes et la conversion en unités Hounsfield (pente/ordonnée DICOM).
Les nodules retenus sont ceux annotés par **au moins 3 radiologues** (regroupement `cluster_annotations`).

In [ ]:
import configparser
configparser.SafeConfigParser = getattr(configparser, 'SafeConfigParser', configparser.ConfigParser)
# Compatibilité pylidc / NumPy récents
for _a, _t in (("int", int), ("float", float), ("bool", bool)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)
rc = Path.home() / (".pylidcrc" if os.name != "nt" else "pylidc.conf")
rc.write_text(f"[dicom]\npath = {CFG['LIDC_ROOT']}\nwarn = True\n")
import pylidc as pl

scans = pl.query(pl.Scan).all()
patients = sorted({s.patient_id for s in scans})
if CFG["FAST"]:
    rng = np.random.default_rng(CFG["SEED"])
    patients = sorted(rng.choice(patients, size=min(CFG["MAX_PATIENTS"], len(patients)), replace=False).tolist())
scan_of = {}
for s in scans:
    if s.patient_id in patients and s.patient_id not in scan_of:
        scan_of[s.patient_id] = s
patients = [p for p in patients if p in scan_of]
print(len(scan_of), "patients retenus")

# --- Téléchargement à la demande depuis le TCIA (API publique NBIA) ---
import io, zipfile, shutil, urllib.request, urllib.parse
NBIA = "https://services.cancerimagingarchive.net/nbia-api"
_TOKEN = {}
def _nbia_token():
    if "v" not in _TOKEN:
        data = urllib.parse.urlencode({"username": "nbia_guest", "password": "", "client_id": "NBIA", "grant_type": "password"}).encode()
        with urllib.request.urlopen(urllib.request.Request(NBIA + "/oauth/token", data=data), timeout=60) as r:
            _TOKEN["v"] = json.load(r)["access_token"]
    return _TOKEN["v"]

def series_dir(pid):
    return Path(CFG["LIDC_ROOT"]) / pid / scan_of[pid].series_instance_uid

def ensure_dicom(pid):
    d = series_dir(pid)
    if (d / ".ok").exists() or not CFG["DOWNLOAD_LIDC"]:
        return d
    uid, last = scan_of[pid].series_instance_uid, None
    for attempt in range(4):
        for url, auth in ((NBIA + "/services/v1/getImage?SeriesInstanceUID=" + uid, False),
                          (NBIA + "/services/v2/getImage?SeriesInstanceUID=" + uid, True)):
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "lidc-tampering-research"})
                if auth: req.add_header("Authorization", "Bearer " + _nbia_token())
                with urllib.request.urlopen(req, timeout=300) as r:
                    data = r.read()
                d.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(io.BytesIO(data)) as z:
                    for m in z.infolist():
                        if m.filename.lower().endswith(".dcm"):
                            (d / Path(m.filename).name).write_bytes(z.read(m))
                (d / ".ok").write_text("ok")
                return d
            except Exception as e:
                last = e
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"téléchargement impossible pour {pid} : {last}")

def drop_dicom(pid):
    if not CFG["KEEP_DICOM"]:
        shutil.rmtree(Path(CFG["LIDC_ROOT"]) / pid, ignore_errors=True)

In [ ]:
def lung_mask_slice(hu):
    # Segmentation pulmonaire simple : air interne (< -400 HU) non connecté au bord, 2 plus grandes composantes.
    m = hu < -400
    lab, n = ndi.label(m)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    m = m & ~np.isin(lab, border)
    lab, n = ndi.label(m)
    if n == 0:
        return m
    sizes = ndi.sum(m, lab, range(1, n + 1))
    order = np.argsort(sizes)[::-1][:2]
    keep = [i + 1 for i in order if sizes[i] > 1500]
    m = np.isin(lab, keep)
    m = ndi.binary_closing(m, iterations=3)
    return ndi.binary_fill_holes(m)

def load_patient(pid):
    ensure_dicom(pid)
    s = scan_of[pid]
    vol = s.to_volume(verbose=False).astype(np.int16)          # (512, 512, Z) en HU
    nods = []
    for cl in s.cluster_annotations():
        if len(cl) >= 3:
            c = np.mean([a.centroid for a in cl], axis=0)       # (ligne, colonne, coupe)
            d = float(np.mean([a.diameter for a in cl]))
            nods.append((int(round(c[0])), int(round(c[1])), int(round(c[2])), d / s.pixel_spacing / 2))
    return vol, nods, float(s.pixel_spacing), float(s.slice_thickness)

vol, nods, ps, st = load_patient(patients[0])
z = nods[0][2] if nods else vol.shape[2] // 2
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(np.clip((vol[:, :, z] + 1350) / 1500, 0, 1), cmap="gray"); ax[0].set_title(f"{patients[0]} coupe {z}")
ax[1].imshow(lung_mask_slice(vol[:, :, z]), cmap="gray"); ax[1].set_title("Masque pulmonaire")
for a in ax: a.axis("off")
plt.show(); print("nodules (≥3 lecteurs) :", nods)

## 2. Partition par patient et inventaire des coupes

Aucune coupe d'un patient du test n'est vue à l'entraînement (coupes d'un même patient = même dose, même noyau, même anatomie).

In [ ]:
rng = np.random.default_rng(CFG["SEED"])
perm = rng.permutation(patients)
n = len(perm); n_tr = int(0.70 * n); n_va = int(0.15 * n)
SPLIT = {p: "train" for p in perm[:n_tr]}
SPLIT.update({p: "val" for p in perm[n_tr:n_tr + n_va]})
SPLIT.update({p: "test" for p in perm[n_tr + n_va:]})
print(pd.Series(SPLIT).value_counts())

def candidate_slices(vol, nods, k, rng):
    # Coupes riches en poumon ; on garantit la présence des coupes à nodule (pour les attaques de suppression).
    Z = vol.shape[2]
    area = np.array([(vol[::4, ::4, z] < -400).mean() for z in range(Z)])
    good = np.where(area > 0.12)[0]
    nod_z = sorted({c[2] for c in nods if 0 <= c[2] < Z})
    rest = [z for z in good if z not in nod_z]
    pick = list(nod_z[: k // 2]) + list(rng.choice(rest, size=min(len(rest), k - min(len(nod_z), k // 2)), replace=False))
    return sorted(int(z) for z in pick)

## 3. Attaques classiques et camouflage du bruit

L'attaquant estime le niveau de bruit local (MAD du résidu médian, éq. 2.1) et ajoute un bruit blanc de même écart-type au contenu inséré, fondu par une fenêtre gaussienne (éq. 2.2).

In [ ]:
HU_MIN, HU_MAX = -1024, 3071

def mad_sigma(img, mask):
    f = img.astype(np.float32)
    r = f - ndi.median_filter(f, 3)
    v = r[mask] if mask.any() else r.ravel()
    return 1.4826 * np.median(np.abs(v - np.median(v))) * 1.2

def blend_patch(img, patch, cy, cx, sigma, rng):
    out = img.astype(np.float32).copy()
    h, w = patch.shape
    y0, x0 = cy - h // 2, cx - w // 2
    yy, xx = np.mgrid[:h, :w]
    win = np.exp(-(((yy - h / 2) / (h / 3.2)) ** 2 + ((xx - w / 2) / (w / 3.2)) ** 2))
    reg = out[y0:y0 + h, x0:x0 + w]
    out[y0:y0 + h, x0:x0 + w] = reg * (1 - win) + (patch + rng.normal(0, sigma, patch.shape)) * win
    m = np.zeros(img.shape, bool); m[y0:y0 + h, x0:x0 + w] = win > 0.25
    return np.clip(np.round(out), HU_MIN, HU_MAX).astype(np.int16), m

def random_lung_point(lung, margin, rng):
    er = ndi.binary_erosion(lung, iterations=margin)
    idx = np.argwhere(er)
    if len(idx) == 0:
        idx = np.argwhere(lung)
    return idx[rng.integers(len(idx))]

def atk_copy_move(img, lung, rng, s=None):
    s = s or int(rng.integers(24, 40))
    a = random_lung_point(lung, s, rng)
    for _ in range(100):
        b = random_lung_point(lung, s, rng)
        if np.abs(a - b).max() > s + 4: break
    out = img.copy()
    out[b[0] - s // 2:b[0] - s // 2 + s, b[1] - s // 2:b[1] - s // 2 + s] = img[a[0] - s // 2:a[0] - s // 2 + s, a[1] - s // 2:a[1] - s // 2 + s]
    m = np.zeros(img.shape, bool); m[b[0] - s // 2:b[0] - s // 2 + s, b[1] - s // 2:b[1] - s // 2 + s] = True
    return out, m

def atk_inpaint(img, lung, nod, rng):
    cy, cx, r = nod
    h = int(2 * r + 16) | 1
    mk = np.zeros(img.shape, np.uint8); cv2.circle(mk, (cx, cy), int(r + 3), 1, -1)
    lo, hi = -1024.0, 400.0
    u8 = np.clip((img.astype(np.float32) - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)
    inp = cv2.inpaint(u8, mk, 5, cv2.INPAINT_TELEA).astype(np.float32) / 255 * (hi - lo) + lo
    patch = ndi.gaussian_filter(inp, 1.0)[cy - h // 2:cy - h // 2 + h, cx - h // 2:cx - h // 2 + h]
    return blend_patch(img, patch, cy, cx, mad_sigma(img, lung), rng)

def atk_noise(img, rng):
    sig = rng.uniform(4, 12)
    body = img > -900
    out = img.astype(np.float32) + rng.normal(0, sig, img.shape) * body
    return np.clip(np.round(out), HU_MIN, HU_MAX).astype(np.int16), body

In [ ]:
# --- FGSM : nécessite un classifieur de nodules (cible de l'attaque adversariale) ---
import timm

def hu_to_net(x):  # normalisation utilisée par tous les réseaux « forensiques » (pas d'écrêtage des poumons)
    return x / 1000.0

class NoduleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = timm.create_model("resnet18", pretrained=True, in_chans=1, num_classes=1)
    def forward(self, x):
        return self.net(x).squeeze(1)

def atk_fgsm(img, lung, model, eps_hu=8.0, P=64):
    # Perturbation FGSM (éq. 2.4) calculée patch par patch sur le poumon, poussant vers « nodule ».
    model.eval()
    out = img.astype(np.float32).copy()
    ys, xs = np.where(lung)
    if len(ys) == 0:
        return img, np.zeros(img.shape, bool)
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    coords = [(y, x) for y in range(y0, max(y0 + 1, y1 - P), P) for x in range(x0, max(x0 + 1, x1 - P), P)]
    batch = np.stack([out[y:y + P, x:x + P] for y, x in coords])[:, None]
    t = torch.tensor(hu_to_net(batch).astype(np.float32), device=DEVICE, requires_grad=True)
    loss = F.binary_cross_entropy_with_logits(model(t), torch.ones(len(coords), device=DEVICE))
    loss.backward()
    g = -t.grad.sign().cpu().numpy()[:, 0]   # descente sur la perte « nodule »
    for (y, x), gi in zip(coords, g):
        out[y:y + P, x:x + P] += eps_hu * gi * lung[y:y + P, x:x + P]
    return np.clip(np.round(out), HU_MIN, HU_MAX).astype(np.int16), lung.copy()

### 3 bis. Cache des coupes candidates et banques de patchs (un seul passage sur les DICOM)

Lecture de chaque patient une seule fois : coupes candidates (HU int16), masques pulmonaires, nodules présents dans la coupe.
Les scanners sont téléchargés par lots de 8 en parallèle pendant que le lot précédent est lu, puis supprimés du disque.
Les patchs servant à entraîner les **outils de l'attaquant** (GAN, DDPM, classifieur cible de FGSM) proviennent **uniquement des patients d'entraînement**.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
P = CFG["PATCH"]
CACHE = {}          # (pid, z) -> dict(img, lung, nods)
NOD_PATCHES, LUNG_PATCHES = [], []
rng = np.random.default_rng(CFG["SEED"] + 1)
t0 = time.time(); FAILED = []
_dl = ThreadPoolExecutor(8)
_fut = {pid: _dl.submit(ensure_dicom, pid) for pid in patients[:8]}
for i, pid in enumerate(patients):
    for nxt in patients[i + 1:i + 9]:                      # pré-téléchargement des 8 suivants
        if nxt not in _fut: _fut[nxt] = _dl.submit(ensure_dicom, nxt)
    try:
        _fut.pop(pid).result()
        vol, nods, ps, st = load_patient(pid)
    except Exception as e:
        print("échec", pid, e); FAILED.append(pid); drop_dicom(pid); continue
    drop_dicom(pid)
    for z in candidate_slices(vol, nods, CFG["SLICES_PER_PATIENT"], rng):
        img = vol[:, :, z].copy()
        lung = lung_mask_slice(img)
        if lung.sum() < 4000:
            continue
        nz = [(r, c, rad) for (r, c, zz, rad) in nods if abs(zz - z) <= 1 and 40 < r < 472 and 40 < c < 472]
        CACHE[(pid, z)] = dict(img=img, lung=lung, nods=nz)
        if SPLIT[pid] == "train":
            for (r, c, rad) in nz:
                NOD_PATCHES.append(img[r - P // 2:r + P // 2, c - P // 2:c + P // 2])
            er_idx = np.argwhere(ndi.binary_erosion(lung, iterations=P // 2))
            for _ in range(6 if len(er_idx) else 0):
                y, x = er_idx[rng.integers(len(er_idx))]
                if all(abs(y - r) > P and abs(x - c) > P for (r, c, _) in nz):
                    pt = img[y - P // 2:y + P // 2, x - P // 2:x + P // 2]
                    if pt.shape == (P, P):
                        LUNG_PATCHES.append(pt)
    if i % 20 == 0:
        print(f"{i}/{len(patients)} patients, {len(CACHE)} coupes, {time.time() - t0:.0f}s", flush=True)
_dl.shutdown()
patients = [p for p in patients if p not in FAILED]
NOD_PATCHES = np.array([p for p in NOD_PATCHES if p.shape == (P, P)], np.int16)
LUNG_PATCHES = np.array(LUNG_PATCHES, np.int16)
# une coupe réelle avec nodule, conservée pour les figures du §12
try:
    np.save("real_slice.npy", CACHE[next(k for k, c in CACHE.items() if c["nods"])]["img"])
except StopIteration:
    pass
print("patients lus :", len(patients), "| échecs :", len(FAILED))
print("coupes en cache :", len(CACHE), "| patchs nodule :", NOD_PATCHES.shape, "| patchs poumon :", LUNG_PATCHES.shape)
chrono("cache des coupes construit")

### 3 ter. Classifieur de nodules (cible de l'attaque FGSM)

In [ ]:
def augment_d4(x):
    k = np.random.randint(4); x = np.rot90(x, k, axes=(-2, -1))
    return np.flip(x, -1) if np.random.rand() < 0.5 else x

Xn = np.concatenate([NOD_PATCHES, LUNG_PATCHES[:len(NOD_PATCHES) * 3]]).astype(np.float32)
yn = np.concatenate([np.ones(len(NOD_PATCHES)), np.zeros(min(len(LUNG_PATCHES), len(NOD_PATCHES) * 3))]).astype(np.float32)
nodcls = NoduleClassifier().to(DEVICE)
opt = torch.optim.Adam(nodcls.parameters(), 1e-4)
for ep in range(CFG["NODCLS_EPOCHS"]):
    perm = np.random.permutation(len(Xn)); tot = 0
    for b in range(0, len(perm), 64):
        idx = perm[b:b + 64]
        xb = torch.tensor(hu_to_net(np.stack([augment_d4(Xn[j]).copy() for j in idx]))[:, None], device=DEVICE)
        loss = F.binary_cross_entropy_with_logits(nodcls(xb), torch.tensor(yn[idx], device=DEVICE))
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
    print(f"époque {ep}: perte {tot / len(perm):.4f}")

## 4. GAN d'inpainting conditionnel (pix2pix) — injection et suppression de nodules

Entrée du générateur : patch $64\times64$ dont le carré central $32\times32$ est masqué, le masque, et une carte de condition (1 = générer un nodule, 0 = générer du parenchyme sain). Perte : $\mathcal{L}_{cGAN} + 100\,\lVert y - G(x_m)\rVert_1$ (éq. 2.5).

In [ ]:
WIN_LO, WIN_HI = -1100.0, 300.0
def to_gan(x):   return (np.clip((x - WIN_LO) / (WIN_HI - WIN_LO), 0, 1) * 2 - 1).astype(np.float32)
def from_gan(y): return (y + 1) / 2 * (WIN_HI - WIN_LO) + WIN_LO

CENTER = np.zeros((P, P), np.float32); CENTER[P // 4:3 * P // 4, P // 4:3 * P // 4] = 1

def cbr(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2, True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2, True))

class UNetG(nn.Module):
    def __init__(self, cin=3, b=48):
        super().__init__()
        self.e1, self.e2, self.e3, self.bt = cbr(cin, b), cbr(b, 2 * b), cbr(2 * b, 4 * b), cbr(4 * b, 8 * b)
        self.u3, self.d3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, 2), cbr(8 * b, 4 * b)
        self.u2, self.d2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, 2), cbr(4 * b, 2 * b)
        self.u1, self.d1 = nn.ConvTranspose2d(2 * b, b, 2, 2), cbr(2 * b, b)
        self.out = nn.Conv2d(b, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2)); bt = self.bt(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(bt), e3], 1)); d2 = self.d2(torch.cat([self.u2(d3), e2], 1)); d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return torch.tanh(self.out(d1))

class PatchD(nn.Module):
    def __init__(self, cin=4, b=64):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(cin, b, 4, 2, 1), nn.LeakyReLU(0.2, True),
                                 nn.Conv2d(b, 2 * b, 4, 2, 1), nn.BatchNorm2d(2 * b), nn.LeakyReLU(0.2, True),
                                 nn.Conv2d(2 * b, 4 * b, 4, 2, 1), nn.BatchNorm2d(4 * b), nn.LeakyReLU(0.2, True),
                                 nn.Conv2d(4 * b, 1, 3, 1, 1))
    def forward(self, x): return self.net(x)

def gan_inputs(y, cond):
    # y : (B,1,P,P) dans [-1,1] ; cond : (B,) 0/1
    m = torch.tensor(CENTER, device=y.device)[None, None].expand_as(y)
    c = cond.view(-1, 1, 1, 1).float().expand_as(y)
    return torch.cat([y * (1 - m), m, c], 1), m

G, D = UNetG().to(DEVICE), PatchD().to(DEVICE)
oG = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5, 0.999)); oD = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5, 0.999))
Xg = np.concatenate([NOD_PATCHES, LUNG_PATCHES]); cg = np.concatenate([np.ones(len(NOD_PATCHES)), np.zeros(len(LUNG_PATCHES))])
w = np.where(cg == 1, len(cg) / (2 * cg.sum()), len(cg) / (2 * (1 - cg).sum())); w /= w.sum()   # rééquilibrage
bce = nn.BCEWithLogitsLoss()
steps = max(1, len(Xg) // 64)
for ep in range(CFG["GAN_EPOCHS"]):
    for _ in range(steps):
        idx = np.random.choice(len(Xg), 64, p=w)
        y = torch.tensor(to_gan(np.stack([augment_d4(Xg[j]).copy() for j in idx]))[:, None], device=DEVICE)
        cond = torch.tensor(cg[idx], device=DEVICE)
        x, m = gan_inputs(y, cond)
        fake = G(x); comp = y * (1 - m) + fake * m
        # discriminateur
        dr, df = D(torch.cat([x, y], 1)), D(torch.cat([x, comp.detach()], 1))
        lD = 0.5 * (bce(dr, torch.ones_like(dr)) + bce(df, torch.zeros_like(df)))
        oD.zero_grad(); lD.backward(); oD.step()
        # générateur
        df = D(torch.cat([x, comp], 1))
        lG = bce(df, torch.ones_like(df)) + 100 * F.l1_loss(fake * m, y * m)
        oG.zero_grad(); lG.backward(); oG.step()
    print(f"GAN époque {ep}: L_D={lD.item():.3f} L_G={lG.item():.3f}")
torch.save(G.state_dict(), OUT / "gan_G.pt")

In [ ]:
@torch.no_grad()
def gan_fill(patch_hu, cond):
    G.eval()
    y = torch.tensor(to_gan(patch_hu)[None, None], device=DEVICE)
    x, m = gan_inputs(y, torch.tensor([cond], device=DEVICE))
    out = (y * (1 - m) + G(x) * m)[0, 0].cpu().numpy()
    return from_gan(out)

def paste_center(img, lung, new_patch, cy, cx, rng):
    # Recolle la zone centrale générée (40x40) avec fenêtre gaussienne + bruit de camouflage.
    c = new_patch[P // 2 - 20:P // 2 + 20, P // 2 - 20:P // 2 + 20]
    return blend_patch(img, c, cy, cx, mad_sigma(img, lung), rng)

def atk_gan_inject(img, lung, rng):
    cy, cx = random_lung_point(lung, P // 2 + 2, rng)
    return paste_center(img, lung, gan_fill(img[cy - P // 2:cy + P // 2, cx - P // 2:cx + P // 2], 1), cy, cx, rng)

def atk_gan_remove(img, lung, nod, rng):
    cy, cx, _ = nod
    return paste_center(img, lung, gan_fill(img[cy - P // 2:cy + P // 2, cx - P // 2:cx + P // 2], 0), cy, cx, rng)

# Contrôle visuel
k = min(4, len(NOD_PATCHES))
fig, ax = plt.subplots(2, k, figsize=(3 * k, 6))
for j in range(k):
    ax[0, j].imshow(to_gan(NOD_PATCHES[j]), cmap="gray"); ax[0, j].set_title("réel (nodule)")
    ax[1, j].imshow(to_gan(gan_fill(NOD_PATCHES[j], 0)), cmap="gray"); ax[1, j].set_title("GAN : suppression")
for a in ax.ravel(): a.axis("off")
plt.show()

## 5. Modèle de diffusion (DDPM) et inpainting de type RePaint

DDPM à $T=1000$ pas (planning linéaire de $\beta_t$), perte $\lVert\epsilon-\epsilon_\theta(x_t,t)\rVert^2$ (éq. 1.7), entraîné **uniquement sur des patchs pulmonaires authentiques**.
**Attaquant et défenseur disposent de deux modèles distincts** (`SEPARATE_DEFENDER_DDPM=True`), entraînés sur deux moitiés disjointes des patients d'entraînement : le défenseur ne connaît donc pas le générateur de l'attaquant, ce qui évite de surestimer la branche diffusion.

In [ ]:
SEPARATE_DEFENDER_DDPM = True
T = CFG["T_DIFF"]
betas = torch.linspace(1e-4, 0.02, T, device=DEVICE)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

class TEmb(nn.Module):
    def __init__(self, d=128):
        super().__init__(); self.d = d
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.SiLU(), nn.Linear(4 * d, 4 * d))
    def forward(self, t):
        half = self.d // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t.float()[:, None] * f[None]
        return self.mlp(torch.cat([a.sin(), a.cos()], 1))

class ResB(nn.Module):
    def __init__(self, i, o, td=512):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, i), nn.Conv2d(i, o, 3, padding=1)
        self.t = nn.Linear(td, o)
        self.n2, self.c2 = nn.GroupNorm(8, o), nn.Conv2d(o, o, 3, padding=1)
        self.sk = nn.Conv2d(i, o, 1) if i != o else nn.Identity()
    def forward(self, x, te):
        h = self.c1(F.silu(self.n1(x))) + self.t(te)[:, :, None, None]
        return self.c2(F.silu(self.n2(h))) + self.sk(x)

class DiffUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.te = TEmb(128); self.inc = nn.Conv2d(1, 64, 3, padding=1)
        self.d1, self.d2, self.d3, self.mid = ResB(64, 64), ResB(64, 128), ResB(128, 256), ResB(256, 256)
        self.u3, self.u2, self.u1 = ResB(512, 128), ResB(256, 64), ResB(128, 64)
        self.no, self.out = nn.GroupNorm(8, 64), nn.Conv2d(64, 1, 3, padding=1)
    def forward(self, x, t):
        te = self.te(t); x0 = self.inc(x)
        h1 = self.d1(x0, te); h2 = self.d2(F.avg_pool2d(h1, 2), te); h3 = self.d3(F.avg_pool2d(h2, 2), te)
        m = self.mid(F.avg_pool2d(h3, 2), te)
        u = self.u3(torch.cat([F.interpolate(m, scale_factor=2), h3], 1), te)
        u = self.u2(torch.cat([F.interpolate(u, scale_factor=2), h2], 1), te)
        u = self.u1(torch.cat([F.interpolate(u, scale_factor=2), h1], 1), te)
        return self.out(F.silu(self.no(u)))

def train_ddpm(patches, epochs, tag):
    net = DiffUNet().to(DEVICE); opt = torch.optim.AdamW(net.parameters(), 2e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    X = to_gan(patches)
    for ep in range(epochs):
        perm = np.random.permutation(len(X)); tot = 0
        for b in range(0, len(perm) - 63, 64):
            x0 = torch.tensor(np.stack([augment_d4(X[j]).copy() for j in perm[b:b + 64]])[:, None], device=DEVICE)
            t = torch.randint(0, T, (len(x0),), device=DEVICE); eps = torch.randn_like(x0)
            xt = abar[t].sqrt()[:, None, None, None] * x0 + (1 - abar[t]).sqrt()[:, None, None, None] * eps
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = F.mse_loss(net(xt, t), eps)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
        print(f"DDPM[{tag}] époque {ep}: perte {tot / max(1, len(perm) // 64):.4f}")
    torch.save(net.state_dict(), OUT / f"ddpm_{tag}.pt")
    return net.eval()

half = len(LUNG_PATCHES) // 2       # les patchs sont rangés patient par patient : les deux moitiés sont disjointes en patients (à un patient près)
ddpm_att = train_ddpm(LUNG_PATCHES[:half] if SEPARATE_DEFENDER_DDPM else LUNG_PATCHES, CFG["DDPM_EPOCHS"], "attaquant")
ddpm_def = train_ddpm(LUNG_PATCHES[half:], CFG["DDPM_EPOCHS"], "defenseur") if SEPARATE_DEFENDER_DDPM else ddpm_att
chrono("DDPM entraînés")

In [ ]:
@torch.no_grad()
def repaint(net, patch_hu, mask):
    # Inpainting RePaint (sans rééchantillonnage) : la région connue est réinjectée bruitée à chaque pas.
    x0 = torch.tensor(to_gan(patch_hu)[None, None], device=DEVICE)
    m = torch.tensor(mask[None, None].astype(np.float32), device=DEVICE)    # 1 = à générer
    x = torch.randn_like(x0)
    for t in reversed(range(T)):
        tt = torch.full((1,), t, device=DEVICE, dtype=torch.long)
        eps = net(x, tt)
        mean = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
        x_unk = mean + betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean
        ab_prev = abar[t - 1] if t > 0 else torch.tensor(1.0, device=DEVICE)
        x_kn = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * torch.randn_like(x0) if t > 0 else x0
        x = m * x_unk + (1 - m) * x_kn
    return from_gan(x[0, 0].clamp(-1, 1).cpu().numpy())

def atk_diffusion(img, lung, rng, nod=None):
    if nod is not None:
        cy, cx, _ = nod
    else:
        cy, cx = random_lung_point(lung, P // 2 + 2, rng)
    new = repaint(ddpm_att, img[cy - P // 2:cy + P // 2, cx - P // 2:cx + P // 2], CENTER)
    return paste_center(img, lung, new, cy, cx, rng)

## 5 bis. Construction du jeu falsifié

Chaque coupe candidate reçoit soit aucune attaque (authentique, probabilité `P_AUTH`), soit **une** attaque tirée parmi celles compatibles (les suppressions exigent un nodule dans la coupe). Le masque de vérité terrain est conservé.

In [ ]:
ATTACKS = ["authentique", "copier_deplacer", "inpainting", "bruit", "fgsm", "gan_injection", "gan_suppression", "diffusion"]
NEEDS_NODULE = {"inpainting", "gan_suppression"}
GLOBAL_ATTACKS = {"bruit", "fgsm"}

def apply_attack(name, img, lung, nods, rng):
    if name == "authentique":     return img, np.zeros(img.shape, bool)
    if name == "copier_deplacer": return atk_copy_move(img, lung, rng)
    if name == "inpainting":      return atk_inpaint(img, lung, nods[0], rng)
    if name == "bruit":           return atk_noise(img, rng)
    if name == "fgsm":            return atk_fgsm(img, lung, nodcls)
    if name == "gan_injection":   return atk_gan_inject(img, lung, rng)
    if name == "gan_suppression": return atk_gan_remove(img, lung, nods[0], rng)
    if name == "diffusion":       return atk_diffusion(img, lung, rng, nods[0] if nods and rng.random() < 0.5 else None)
    raise ValueError(name)

rng = np.random.default_rng(CFG["SEED"] + 2)
DATA = {}   # (pid, z) -> dict(img, mask, lung, attack, split)
rows = []; ATK_FAIL = {}
for (pid, z), c in CACHE.items():
    if rng.random() < CFG["P_AUTH"]:
        atk = "authentique"
    else:
        pool = [a for a in ATTACKS[1:] if (a not in NEEDS_NODULE) or c["nods"]]
        atk = pool[rng.integers(len(pool))]
    try:
        out, m = apply_attack(atk, c["img"], c["lung"], c["nods"], rng)
    except Exception as e:     # bord d'image, etc. -> coupe laissée authentique
        ATK_FAIL[atk] = ATK_FAIL.get(atk, 0) + 1
        atk, out, m = "authentique", c["img"], np.zeros(c["img"].shape, bool)
    DATA[(pid, z)] = dict(img=out, mask=m, lung=c["lung"], attack=atk, split=SPLIT[pid])
    rows.append(dict(pid=pid, z=z, split=SPLIT[pid], attack=atk, label=int(atk != "authentique"), area=float(m.mean())))
MANIFEST = pd.DataFrame(rows)
MANIFEST.to_csv(OUT / "manifest.csv", index=False)
print(pd.crosstab(MANIFEST.attack, MANIFEST.split))
print("attaques échouées (coupes laissées authentiques) :", ATK_FAIL)
CACHE.clear()      # libère la mémoire : DATA contient désormais toutes les coupes
chrono("jeu falsifié construit")

## 6. Branche forensique

Le module `features.py` (identique à celui de la preuve de concept) fournit les résidus SRM/KV, l'autocorrélation (éq. 2.6), l'énergie spectrale par anneaux (éq. 2.7), la normalisation robuste intra-coupe (éq. 2.8) et la détection de duplication par hachage.
On l'applique ici sur des **patchs $64\times64$ au pas de 32** couvrant les poumons.

In [ ]:
%%writefile features.py
"""Extraction de caractéristiques forensiques (résidus de bruit, spectre, cohérence locale, duplication)."""
import numpy as np
from scipy import ndimage as ndi
from scipy.stats import kurtosis, skew

B = 16
K_SRM3 = np.array([[-1, 2, -1], [2, -4, 2], [-1, 2, -1]], np.float32) / 4.0
K_KV = np.array([[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2],
                 [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]], np.float32) / 12.0


def residuals(img):
    f = img.astype(np.float32)
    return {
        "med": f - ndi.median_filter(f, 3),
        "srm3": ndi.convolve(f, K_SRM3),
        "kv": ndi.convolve(f, K_KV),
        "dx": np.diff(f, axis=1, append=f[:, -1:]),
    }


def ac1(r):
    a = r - r.mean()
    d = (a * a).mean() + 1e-6
    return float((a[:, 1:] * a[:, :-1]).mean() / d), float((a[1:] * a[:-1]).mean() / d)


def band_energy(r, nb=6):
    F = np.abs(np.fft.fftshift(np.fft.fft2(r - r.mean()))) ** 2
    h, w = r.shape
    yy, xx = np.mgrid[:h, :w]
    rad = np.sqrt((yy - h / 2) ** 2 + (xx - w / 2) ** 2) / (min(h, w) / 2)
    e = np.array([F[(rad >= i / nb) & (rad < (i + 1) / nb)].mean() for i in range(nb)])
    return e / (e.sum() + 1e-9)


def lung_blocks(lungs):
    er = ndi.binary_erosion(lungs, iterations=2)
    out = []
    for y in range(0, lungs.shape[0] - B + 1, B // 2):
        for x in range(0, lungs.shape[1] - B + 1, B // 2):
            if er[y:y + B, x:x + B].all():
                out.append((y, x))
    return out


def block_feats(img, R, blocks):
    """Descripteurs locaux par bloc : niveau de bruit, autocorrélation, ratio spectral haut/moyen."""
    rows = []
    for y, x in blocks:
        r = R["med"][y:y + B, x:x + B]
        k = R["kv"][y:y + B, x:x + B]
        sig = 1.4826 * np.median(np.abs(r - np.median(r)))
        a1x, a1y = ac1(k)
        e = band_energy(k, 4)
        rows.append([sig, a1x, a1y, e[3] / (e[1] + 1e-9), k.std(), kurtosis(k.ravel())])
    return np.array(rows, np.float32)


def duplicate_map(img, body, b=8):
    """Détection de blocs strictement identiques (copier-déplacer) par hachage."""
    from numpy.lib.stride_tricks import sliding_window_view
    dup = np.zeros(img.shape, bool)
    W = sliding_window_view(np.ascontiguousarray(img), (b, b))
    H, Wd = W.shape[:2]
    flat = np.ascontiguousarray(W.reshape(H * Wd, b * b))
    yy, xx = np.divmod(np.arange(H * Wd), Wd)
    ok = body[yy + b // 2, xx + b // 2] & (flat.std(1) >= 3)
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return dup
    v = flat[idx].view(np.dtype((np.void, flat.dtype.itemsize * b * b))).ravel()
    _, inv, cnt = np.unique(v, return_inverse=True, return_counts=True)
    for g in np.where(cnt > 1)[0]:
        mem = idx[inv == g]
        ys, xs = yy[mem], xx[mem]
        if (np.abs(ys - ys[0]).max() > b) or (np.abs(xs - xs[0]).max() > b):
            for y, x in zip(ys, xs):
                dup[y:y + b, x:x + b] = True
    return dup


def robust_z(v):
    med = np.median(v, 0)
    mad = 1.4826 * np.median(np.abs(v - med), 0) + 1e-6
    return (v - med) / mad


def image_features(img, lungs):
    R = residuals(img)
    body = img > -900
    feats = []
    for k in ("med", "srm3", "kv", "dx"):
        v = R[k][body]
        feats += [v.std(), kurtosis(v), skew(v)]
    lv = R["kv"][lungs]
    feats += list(ac1(np.where(lungs, R["kv"], 0)))
    # spectre du résidu dans une fenêtre pulmonaire centrale
    ys, xs = np.where(ndi.binary_erosion(lungs, iterations=4))
    cy, cx = (int(np.median(ys)), int(np.median(xs))) if len(ys) else (img.shape[0] // 2, img.shape[1] // 2)
    win = R["kv"][max(cy - 32, 0):cy + 32, max(cx - 32, 0):cx + 32]
    feats += list(band_energy(win, 6))
    blocks = lung_blocks(lungs)
    bf = block_feats(img, R, blocks)
    if len(bf) == 0:
        bf = np.zeros((1, 6), np.float32)
    z = np.abs(robust_z(bf))
    feats += list(z.max(0)) + list(np.percentile(z, 95, axis=0)) + list(bf.std(0) / (np.abs(bf.mean(0)) + 1e-6))
    dup = duplicate_map(img, body)
    feats += [dup.mean() * 1000, float(dup.any())]
    return np.array(feats, np.float32), (blocks, bf, dup)


def intensity_features(img, lungs):
    """Ligne de base « contenu » : statistiques d'intensité HU uniquement."""
    v = img[lungs].astype(np.float32)
    b = img[img > -900].astype(np.float32)
    return np.array([v.mean(), v.std(), *np.percentile(v, [5, 25, 50, 75, 95]),
                     b.mean(), b.std(), *np.histogram(b, bins=12, range=(-1000, 1000), density=True)[0]], np.float32)

In [ ]:
import importlib, features as FZ
importlib.reload(FZ)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, accuracy_score, precision_score, recall_score

S = CFG["STRIDE"]

def grid_coords(lung):
    out = []
    for y in range(0, lung.shape[0] - P + 1, S):
        for x in range(0, lung.shape[1] - P + 1, S):
            if lung[y:y + P, x:x + P].mean() > 0.5:
                out.append((y, x))
    return out

def patch_desc(R, img, y, x):
    r = R["med"][y:y + P, x:x + P]; k = R["kv"][y:y + P, x:x + P]
    sig = 1.4826 * np.median(np.abs(r - np.median(r)))
    a1x, a1y = FZ.ac1(k); e = FZ.band_energy(k, 6)
    blk = img[y:y + P, x:x + P].astype(np.float32)
    return [sig, a1x, a1y, e[5] / (e[2] + 1e-9), k.std(), kurtosis(k.ravel()), *e, blk.mean(), blk.std()]

def slice_patch_features(img, lung, coords=None):
    coords = coords if coords is not None else grid_coords(lung)
    if not coords:
        return coords, np.zeros((0, 27), np.float32)
    R = FZ.residuals(img)
    f = np.array([patch_desc(R, img, y, x) for y, x in coords], np.float32)
    z = np.abs(FZ.robust_z(f[:, :12]))
    dup = FZ.duplicate_map(img, img > -900)
    d = np.array([[dup[y:y + P, x:x + P].mean()] for y, x in coords], np.float32)
    return coords, np.nan_to_num(np.hstack([f, z, d]))

def patch_labels(mask, coords, attack):
    if attack in GLOBAL_ATTACKS:
        return np.ones(len(coords), int)
    return np.array([mask[y:y + P, x:x + P].mean() > 0.3 for y, x in coords], int)

# Extraction pour toutes les coupes (parallélisée sur les cœurs CPU), avec descripteurs globaux de la coupe
def _pf_job(key):
    d = DATA[key]
    coords, f = slice_patch_features(d["img"], d["lung"])
    return coords, f, FZ.image_features(d["img"], d["lung"])[0]

t0 = time.time()
_keys = list(DATA)
PF = {}
for key, (coords, f, g) in zip(_keys, pmap(_pf_job, _keys)):
    PF[key] = dict(coords=coords, f=f, g=g, y=patch_labels(DATA[key]["mask"], coords, DATA[key]["attack"]))
print(f"descripteurs de patchs : {time.time() - t0:.0f}s pour {len(PF)} coupes")

def stack_split(split):
    keys = [k for k in PF if DATA[k]["split"] == split and len(PF[k]["coords"])]
    X = np.vstack([PF[k]["f"] for k in keys]); y = np.concatenate([PF[k]["y"] for k in keys])
    owner = np.concatenate([[i] * len(PF[k]["coords"]) for i, k in enumerate(keys)])
    return keys, X, y, owner

trk, Xtr, ytr, _ = stack_split("train")
# sous-échantillonnage des négatifs (3:1)
neg = np.where(ytr == 0)[0]; pos = np.where(ytr == 1)[0]
sel = np.concatenate([pos, np.random.choice(neg, min(len(neg), 3 * len(pos)), replace=False)])
forensic_clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, class_weight="balanced", random_state=0)
forensic_clf.fit(Xtr[sel], ytr[sel])
vak, Xva, yva, _ = stack_split("val")
print("AUC patchs (val) — branche forensique :", round(roc_auc_score(yva, forensic_clf.predict_proba(Xva)[:, 1]), 4))
chrono("branche forensique entraînée")

## 7. Branche CNN : couche contrainte de Bayar + SRM → EfficientNet-B0 → décodeur U-Net

* Couche de Bayar (éq. 1.9) : poids centraux fixés à −1, somme des voisins = 1, reprojection à chaque passe avant.
* 3 résidus SRM fixes (KV $5\times5$, SRM $3\times3$, différence horizontale) + l'image (HU/1000) → 7 canaux.
* Perte : BCE (patch) + Dice (carte), éq. 2.9. Augmentation : D4, flou, ré-échantillonnage, bruit, export 8 bits.

In [ ]:
class BayarConv(nn.Module):
    def __init__(self, cin=1, cout=3, k=5):
        super().__init__(); self.k = k
        self.w = nn.Parameter(torch.randn(cout, cin, k, k) * 0.01)
        m = torch.ones(cout, cin, k, k); m[:, :, k // 2, k // 2] = 0
        c = torch.zeros(cout, cin, k, k); c[:, :, k // 2, k // 2] = -1
        self.register_buffer("mask", m); self.register_buffer("center", c)
    def forward(self, x):
        w = self.w * self.mask
        w = w / (w.sum(dim=(2, 3), keepdim=True) + 1e-8) + self.center
        return F.conv2d(x, w, padding=self.k // 2)

def srm_bank():
    kv = torch.tensor(FZ.K_KV); s3 = torch.zeros(5, 5); s3[1:4, 1:4] = torch.tensor(FZ.K_SRM3)
    d1 = torch.zeros(5, 5); d1[2, 2], d1[2, 3] = -1, 1
    return torch.stack([kv, s3, d1])[:, None]

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(cin + cskip, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(True),
                                  nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(True))
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], 1))

class ForensicUNet(nn.Module):
    def __init__(self, encoder="efficientnet_b0", pretrained=True):
        super().__init__()
        self.bayar = BayarConv(1, 3, 5)
        self.register_buffer("srm", srm_bank())
        self.enc = timm.create_model(encoder, features_only=True, pretrained=pretrained, in_chans=7)
        ch = self.enc.feature_info.channels()
        self.cls = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(ch[-1], 1))
        self.ups = nn.ModuleList()
        cur = ch[-1]
        for c in reversed(ch[:-1]):
            self.ups.append(Up(cur, c, max(32, c))); cur = max(32, c)
        self.seg = nn.Conv2d(cur, 1, 1)
    def forward(self, x):
        x7 = torch.cat([x, self.bayar(x), F.conv2d(x, self.srm, padding=2)], 1)
        feats = self.enc(x7)
        logit = self.cls(feats[-1]).squeeze(1)
        d = feats[-1]
        for up, sk in zip(self.ups, reversed(feats[:-1])):
            d = up(d, sk)
        seg = F.interpolate(self.seg(d), size=x.shape[-2:], mode="bilinear", align_corners=False)
        return logit, seg[:, 0]

def postprocess(img, kind):
    f = img.astype(np.float32)
    if kind == "blur":    f = ndi.gaussian_filter(f, 0.5)
    elif kind == "resample":
        h, w = f.shape; f = cv2.resize(cv2.resize(f, (int(w * 0.75), int(h * 0.75)), interpolation=cv2.INTER_LINEAR), (w, h), interpolation=cv2.INTER_LINEAR)
    elif kind == "noise": f = f + np.random.normal(0, 2, f.shape)
    elif kind == "8bit":
        lo, hi = -1350, 150; f = np.round(np.clip((f - lo) / (hi - lo), 0, 1) * 255) / 255 * (hi - lo) + lo
    return np.clip(np.round(f), HU_MIN, HU_MAX).astype(np.int16)

class PatchSet(Dataset):
    def __init__(self, split, neg_ratio=2, augment=False):
        self.items = []
        for k, pf in PF.items():
            if DATA[k]["split"] != split: continue
            for (y, x), lab in zip(pf["coords"], pf["y"]):
                self.items.append((k, y, x, int(lab)))
        self.augment = augment; self.neg_ratio = neg_ratio; self.resample()
    def resample(self):
        pos = [i for i in self.items if i[3] == 1]; neg = [i for i in self.items if i[3] == 0]
        random.shuffle(neg); self.cur = pos + neg[:self.neg_ratio * max(1, len(pos))]; random.shuffle(self.cur)
    def __len__(self): return len(self.cur)
    def __getitem__(self, i):
        k, y, x, lab = self.cur[i]; d = DATA[k]
        y0, x0 = max(0, y - 8), max(0, x - 8)          # marge pour que flou / ré-échantillonnage n'aient pas d'effet de bord
        big = d["img"][y0:y + P + 8, x0:x + P + 8]; mbig = d["mask"][y0:y + P + 8, x0:x + P + 8]
        if self.augment and random.random() < 0.4:
            big = postprocess(big, random.choice(["blur", "resample", "noise", "8bit"]))
        oy, ox = y - y0, x - x0
        pt = big[oy:oy + P, ox:ox + P].astype(np.float32); m = mbig[oy:oy + P, ox:ox + P].astype(np.float32)
        if d["attack"] in GLOBAL_ATTACKS: m = np.ones_like(m) * lab
        if self.augment:
            kk = random.randint(0, 3); pt, m = np.rot90(pt, kk), np.rot90(m, kk)
            if random.random() < 0.5: pt, m = pt[:, ::-1], m[:, ::-1]
        return torch.tensor(hu_to_net(pt.copy()))[None], torch.tensor(float(lab)), torch.tensor(m.copy())

def dice_loss(seg_logit, m, eps=1.0):
    p = torch.sigmoid(seg_logit)
    num = 2 * (p * m).sum((1, 2)) + eps; den = p.sum((1, 2)) + m.sum((1, 2)) + eps
    return (1 - num / den).mean()

cnn = ForensicUNet().to(DEVICE)
opt = torch.optim.Adam(cnn.parameters(), CFG["LR"])
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
tr_set, va_set = PatchSet("train", augment=True), PatchSet("val", neg_ratio=3)
best, patience = -1, 0
for ep in range(CFG["CNN_EPOCHS"]):
    cnn.train(); tr_set.resample()
    for xb, yb, mb in DataLoader(tr_set, CFG["BATCH"], shuffle=True, num_workers=NUM_WORKERS, drop_last=True):
        xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            logit, seg = cnn(xb)
            loss = F.binary_cross_entropy_with_logits(logit, yb) + dice_loss(seg.float(), mb)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    cnn.eval(); ps_, ys_ = [], []
    with torch.no_grad():
        for xb, yb, _ in DataLoader(va_set, 256, num_workers=NUM_WORKERS):
            ps_.append(torch.sigmoid(cnn(xb.to(DEVICE))[0]).cpu().numpy()); ys_.append(yb.numpy())
    auc = roc_auc_score(np.concatenate(ys_), np.concatenate(ps_))
    print(f"CNN époque {ep}: perte {loss.item():.4f} | AUC patchs val {auc:.4f}")
    if auc > best:
        best, patience = auc, 0; torch.save(cnn.state_dict(), OUT / "cnn_best.pt")
    else:
        patience += 1
        if patience >= 6: print("arrêt précoce"); break
cnn.load_state_dict(torch.load(OUT / "cnn_best.pt")); cnn.eval()
chrono(f"branche CNN entraînée (meilleure AUC patchs val {best:.4f})")

## 8. Branche diffusion : score d'erreur de reconstruction (éq. 2.10)

Pour chaque patch : bruitage jusqu'à $t^\ast$, prédiction de $\hat x_0$ en un pas, erreur quadratique moyenne. Branche **non supervisée** (aucun exemple falsifié utilisé).
L'erreur est normalisée au sein de la coupe (score z robuste) ; une région falsifiée peut se reconstruire **moins bien** (contenu étranger) ou **trop bien** (contenu généré, cf. DIRE) : le score de coupe utilise donc l'écart absolu $|z|$.

In [ ]:
@torch.no_grad()
def diffusion_score(patches_hu, net=None, t_star=None, n_draws=3, bs=256):
    net = net or ddpm_def; t_star = t_star or CFG["T_STAR"]
    out = []
    for b in range(0, len(patches_hu), bs):
        x0 = torch.tensor(to_gan(patches_hu[b:b + bs])[:, None], device=DEVICE)
        t = torch.full((len(x0),), t_star, device=DEVICE, dtype=torch.long)
        acc = torch.zeros(len(x0), device=DEVICE)
        for _ in range(n_draws):
            eps = torch.randn_like(x0)
            xt = abar[t_star].sqrt() * x0 + (1 - abar[t_star]).sqrt() * eps
            x0h = (xt - (1 - abar[t_star]).sqrt() * net(xt, t)) / abar[t_star].sqrt()
            acc += ((x0h - x0) ** 2).mean((1, 2, 3))
        out.append((acc / n_draws).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

## 9. Fusion local → global et évaluation

* **Local** : pour chaque patch, $[p_{for}, p_{cnn}, z(s_{diff})]$ → régression logistique (apprise sur la validation).
* **Global** : agrégation (max, moyenne top-3, top-10, nombre > 0,5) + descripteurs globaux de la coupe → HGB (appris sur la validation), éq. 2.11.
* Test interne (patients jamais vus), **test externe CT-GAN** (générateur jamais vu) et robustesse aux post-traitements.

In [ ]:
@torch.no_grad()
def cnn_patch(img, coords):
    if not coords: return np.zeros(0), np.zeros((0, P, P))
    X = np.stack([img[y:y + P, x:x + P] for y, x in coords]).astype(np.float32)
    pr, sg = [], []
    for b in range(0, len(X), 256):
        lo, se = cnn(torch.tensor(hu_to_net(X[b:b + 256]))[:, None].to(DEVICE))
        pr.append(torch.sigmoid(lo).cpu().numpy()); sg.append(torch.sigmoid(se).cpu().numpy())
    return np.concatenate(pr), np.concatenate(sg)

def cpu_part(img, lung):
    coords, f = slice_patch_features(img, lung)
    return coords, f, FZ.image_features(img, lung)[0]

def gpu_part(img, coords, f, gfeat, keep_seg=True):
    if not coords:
        return None
    p_for = forensic_clf.predict_proba(f)[:, 1]
    p_cnn, seg = cnn_patch(img, coords)
    X = np.stack([img[y:y + P, x:x + P] for y, x in coords])
    s = diffusion_score(X)
    s_z = (s - np.median(s)) / (1.4826 * np.median(np.abs(s - np.median(s))) + 1e-9)
    return dict(coords=coords, p_for=p_for, p_cnn=p_cnn, seg=seg if keep_seg else None, s_diff=s_z, gfeat=gfeat)

def branch_outputs(img, lung):
    return gpu_part(img, *cpu_part(img, lung))

_JOB = {}
def _cpu_job(key):
    img, lung = _JOB[key]
    return cpu_part(img, lung)

def branch_outputs_many(images, keep_seg=True):
    # images : dict clé -> (img, lung). Partie CPU en parallèle, partie GPU séquentielle.
    global _JOB
    _JOB = images; keys = list(images)
    cpu = pmap(_cpu_job, keys)
    out = {k: gpu_part(images[k][0], *c, keep_seg=keep_seg) for k, c in zip(keys, cpu)}
    _JOB = {}
    return {k: v for k, v in out.items() if v is not None}

def agg(p):
    p = np.sort(p)[::-1]
    return [p[0], p[:3].mean(), p[:10].mean(), (p > 0.5).sum()] if len(p) else [0, 0, 0, 0]

def pixel_map(coords, vals, shape=(512, 512)):
    acc = np.zeros(shape); cnt = np.zeros(shape)
    for (y, x), v in zip(coords, vals):
        acc[y:y + P, x:x + P] += v; cnt[y:y + P, x:x + P] += 1
    return np.where(cnt > 0, acc / np.maximum(cnt, 1), 0)

# descripteurs CPU déjà calculés au §6 : seule la partie GPU est exécutée ici
BO = {}
for k, d in DATA.items():
    if d["split"] in ("val", "test"):
        bo = gpu_part(d["img"], PF[k]["coords"], PF[k]["f"], PF[k]["g"])
        if bo is not None: BO[k] = bo
chrono("sorties des trois branches calculées (validation + test)")

# --- apprentissage de la fusion sur la VALIDATION uniquement ---
vkeys = [k for k in BO if DATA[k]["split"] == "val"]
L_X = np.vstack([np.c_[BO[k]["p_for"], BO[k]["p_cnn"], BO[k]["s_diff"], np.abs(BO[k]["s_diff"])] for k in vkeys])
L_y = np.concatenate([PF[k]["y"] for k in vkeys])
local_fuser = LogisticRegression(max_iter=2000, class_weight="balanced").fit(L_X, L_y)

def fused_local(bo):
    return local_fuser.predict_proba(np.c_[bo["p_for"], bo["p_cnn"], bo["s_diff"], np.abs(bo["s_diff"])])[:, 1]

def meta_features(bo):
    return np.r_[bo["gfeat"], agg(fused_local(bo)), agg(bo["p_for"]), agg(bo["p_cnn"]), agg(bo["s_diff"]), agg(np.abs(bo["s_diff"]))]

meta = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0)
meta.fit(np.nan_to_num(np.array([meta_features(BO[k]) for k in vkeys])), np.array([int(DATA[k]["attack"] != "authentique") for k in vkeys]))

def slice_scores(bo):
    return dict(forensique=np.mean(np.sort(bo["p_for"])[::-1][:3]), cnn=np.mean(np.sort(bo["p_cnn"])[::-1][:3]),
                diffusion=np.mean(np.sort(np.abs(bo["s_diff"]))[::-1][:3]),
                hybride=meta.predict_proba(np.nan_to_num(meta_features(bo))[None])[0, 1])

def tpr_at(y, s, fpr_target=0.01):
    fpr, tpr, _ = roc_curve(y, s); return float(np.interp(fpr_target, fpr, tpr))

def dice(pred, gt):
    return 2 * (pred & gt).sum() / (pred.sum() + gt.sum() + 1e-9)

In [ ]:
# --- Test interne ---
tkeys = [k for k in BO if DATA[k]["split"] == "test"]
yt = np.array([int(DATA[k]["attack"] != "authentique") for k in tkeys])
SC = {b: np.array([slice_scores(BO[k])[b] for k in tkeys]) for b in ("forensique", "cnn", "diffusion", "hybride")}
RES = {}
for b, s in SC.items():
    RES[b] = dict(auc=roc_auc_score(yt, s), tpr1=tpr_at(yt, s))
# seuil hybride = 0,5 ; F1 / exactitude
pred_h = (SC["hybride"] >= 0.5).astype(int)
RES["hybride"].update(acc=accuracy_score(yt, pred_h), prec=precision_score(yt, pred_h), rec=recall_score(yt, pred_h), f1=f1_score(yt, pred_h))
# Dice par branche (attaques locales)
for b, getter in (("forensique", lambda bo: bo["p_for"]), ("cnn", None), ("hybride", lambda bo: fused_local(bo))):
    ds = []
    for k in tkeys:
        d = DATA[k]
        if d["attack"] in GLOBAL_ATTACKS or d["attack"] == "authentique": continue
        bo = BO[k]
        if b == "cnn":
            acc = np.zeros((512, 512)); cnt = np.zeros((512, 512))
            for (y, x), sg in zip(bo["coords"], bo["seg"]):
                acc[y:y + P, x:x + P] += sg; cnt[y:y + P, x:x + P] += 1
            pm = np.where(cnt > 0, acc / np.maximum(cnt, 1), 0)
        else:
            pm = pixel_map(bo["coords"], getter(bo))
        ds.append(dice(pm >= 0.5, d["mask"] & d["lung"]))
    RES[b]["dice"] = float(np.mean(ds)) if ds else float("nan")
# détection par type d'attaque (hybride)
per_attack = pd.Series({a: float(((SC["hybride"] >= 0.5) == (a != "authentique"))[np.array([DATA[k]["attack"] == a for k in tkeys])].mean())
                        for a in ATTACKS if any(DATA[k]["attack"] == a for k in tkeys)})
# AUC par type d'attaque (authentiques du test + coupes de ce type), pour chaque branche
AUC_ATK = {}
auth_idx = np.array([DATA[k]["attack"] == "authentique" for k in tkeys])
for a in ATTACKS[1:]:
    sel = np.array([DATA[k]["attack"] == a for k in tkeys])
    if sel.sum() >= 3 and auth_idx.sum() >= 3:
        m_ = sel | auth_idx
        AUC_ATK[a] = {b: float(roc_auc_score(yt[m_], SC[b][m_])) for b in SC}
print(pd.DataFrame(RES).T.round(3)); print(per_attack.round(3)); print(pd.DataFrame(AUC_ATK).T.round(3))
chrono("test interne terminé")

In [ ]:
# --- Test externe : jeu CT-GAN de l'UCI (générateur jamais vu) ---
# Le jeu fournit des scanners DICOM et un CSV de vérité terrain (type, uuid, slice, x, y).
# Types : FM = cancer injecté (faux malin), FB = cancer retiré (faux bénin), TM/TB = authentiques.
# Vérifiez les noms de colonnes et la convention d'indice de coupe en affichant quelques cas.
import pydicom
CT_ROOTS = [Path(CFG["CTGAN_ROOT"])] + [Path(p).parent for p in glob.glob("/kaggle/input/**/labels_exp1.csv", recursive=True)]
def _find_labels():
    out = []
    for root in CT_ROOTS:
        out += glob.glob(str(root / "**" / "labels_exp*.csv"), recursive=True)
    return sorted(set(out))

if not _find_labels() and CFG["DOWNLOAD_CTGAN"]:
    # Archive officielle (UCI id=520, Google Drive, ~6 Go) : téléchargée, extraite puis supprimée.
    try:
        import gdown, zipfile
        root = Path(CFG["CTGAN_ROOT"]); root.mkdir(parents=True, exist_ok=True)
        zp = root / "uci520.zip"
        os.system(f'wget -q -O "{zp}" "https://archive.ics.uci.edu/static/public/520/deepfakes+medical+image+tamper+detection.zip"')
        if not zp.exists() or zp.stat().st_size < 10**8:
            gdown.download(id="1R0WD_5IZ3NlyCiOPf1Ex74nBnZYQegwr", output=str(zp), quiet=True)
        with zipfile.ZipFile(zp) as z: z.extractall(root)
        zp.unlink()
        for inner in list(root.glob("**/*.zip")):
            with zipfile.ZipFile(inner) as z2: z2.extractall(inner.parent)
            inner.unlink()
        print("CT-GAN extrait :", len(list(root.glob("**/labels_exp*.csv"))), "fichiers de labels")
    except Exception as e:
        print("téléchargement CT-GAN impossible :", e)

def load_ctgan_scan(folder):
    # Même convention que le code de CT-GAN : coupes triées par ImagePositionPatient[2] croissant, valeurs en HU.
    sl = []
    for f in glob.glob(str(Path(folder) / "*")):
        try:
            sl.append(pydicom.dcmread(f))
        except Exception:
            pass
    sl = [s for s in sl if hasattr(s, "ImagePositionPatient") and hasattr(s, "PixelData")]
    sl.sort(key=lambda s: float(s.ImagePositionPatient[2]))
    return np.stack([dicom_to_hu_raw(s) for s in sl])

def dicom_to_hu_raw(ds):
    return (ds.pixel_array.astype(np.float32) * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))).astype(np.int16)

EXT = None
labs = _find_labels()
if labs:
    ys_, ss_, ex_imgs, ex_y = [], {b: [] for b in SC}, {}, {}
    for lf in labs:
        lab = pd.read_csv(lf); lab.columns = [c.strip().lower() for c in lab.columns]
        base = Path(lf).parent
        for uuid, grp in lab.groupby("uuid"):
            folder = next(iter(glob.glob(str(base / "**" / str(uuid)), recursive=True)), None)
            if folder is None: continue
            try:
                arr = load_ctgan_scan(folder)
            except Exception as e:
                print("scan illisible", uuid, e); continue
            for j, r in grp.reset_index().iterrows():
                z = int(r["slice"])
                if not 0 <= z < len(arr): continue
                img = arr[z]
                ex_imgs[(lf, uuid, j)] = (img, lung_mask_slice(img))
                ex_y[(lf, uuid, j)] = int(str(r["type"]).strip().upper() in ("FM", "FB"))
    EXB = branch_outputs_many(ex_imgs, keep_seg=False)
    for k, bo in EXB.items():
        sc = slice_scores(bo)
        for b in SC: ss_[b].append(sc[b])
        ys_.append(ex_y[k])
    ys_ = np.array(ys_)
    if len(set(ys_)) == 2:
        EXT = {b: dict(auc=roc_auc_score(ys_, np.array(v)), tpr1=tpr_at(ys_, np.array(v)), n=len(ys_)) for b, v in ss_.items()}
        print(pd.DataFrame(EXT).T.round(3))
else:
    print("Jeu CT-GAN indisponible : test externe non réalisé.")
chrono("test externe CT-GAN terminé")

In [ ]:
# --- Robustesse aux post-traitements (hybride, test interne) ---
ROB = {"aucun": RES["hybride"]["auc"]}
for kind in ("blur", "resample", "noise", "8bit"):
    np.random.seed(0)
    imgs = {k: (postprocess(DATA[k]["img"], kind), DATA[k]["lung"]) for k in tkeys}
    bos = branch_outputs_many(imgs, keep_seg=False)
    s = [slice_scores(bos[k])["hybride"] if k in bos else 0.0 for k in tkeys]
    ROB[kind] = roc_auc_score(yt, s)
print(pd.Series(ROB).round(3))
chrono("robustesse évaluée")

In [ ]:
# --- Génération automatique du tableau 3.6 du mémoire ---
names = {"forensique": "Forensique (blocs + agrégation)", "cnn": "CNN contraint + EfficientNet-B0 / U-Net",
         "diffusion": "Diffusion (erreur de reconstruction)", "hybride": "Hybride (fusion des trois branches)"}
fmt = lambda v: "—" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}".replace(".", ",")
T36 = pd.DataFrame([{
    "Branche": names[b],
    "AUC test interne": fmt(RES[b]["auc"]),
    "AUC CT-GAN (externe)": fmt(EXT[b]["auc"]) if EXT else "non disponible",
    "TPR à FPR = 1 %": fmt(RES[b]["tpr1"]),
    "Dice moyen": fmt(RES[b].get("dice")) if b != "diffusion" else "—"} for b in names])
print(T36.to_markdown(index=False))
T36.to_csv(OUT / "tableau_3_6.csv", index=False)
INFO = dict(patients=len(patients), coupes=len(DATA), coupes_test=len(tkeys), falsifiees_test=int(yt.sum()),
            separate_defender_ddpm=SEPARATE_DEFENDER_DDPM, gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU",
            duree_min=round((time.time() - T_START) / 60, 1))
json.dump(dict(INFO=INFO, RES=RES, EXT=EXT, ROB=ROB, per_attack=per_attack.to_dict(), auc_par_attaque=AUC_ATK,
               attaques_echouees=ATK_FAIL), open(OUT / "resultats_lidc.json", "w"), indent=1, default=float)
print(INFO)

### Visualisation : cartes de localisation sur le test

In [ ]:
ex = [k for k in tkeys if DATA[k]["attack"] not in GLOBAL_ATTACKS | {"authentique"}][:4]
fig, ax = plt.subplots(len(ex), 3, figsize=(10, 3.4 * len(ex)))
for i, k in enumerate(ex):
    d, bo = DATA[k], BO[k]; w = np.clip((d["img"] + 1350) / 1500, 0, 1)
    ax[i, 0].imshow(w, cmap="gray"); ax[i, 0].set_title(f"{d['attack']} — score {slice_scores(bo)['hybride']:.2f}")
    ax[i, 1].imshow(w, cmap="gray"); ax[i, 1].contour(d["mask"], [0.5], colors="orange"); ax[i, 1].set_title("vérité terrain")
    ax[i, 2].imshow(w, cmap="gray"); ax[i, 2].imshow(np.ma.masked_less(pixel_map(bo["coords"], fused_local(bo)), 0.05), cmap="magma", vmin=0, vmax=1, alpha=0.7)
    ax[i, 2].set_title("carte fusionnée")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "localisation_lidc.png", dpi=150); plt.show()

## 10. Module de sécurité pour les hôpitaux

1. Empreinte **SHA-256** des pixels + attributs critiques, comparée à l'empreinte enregistrée à l'acquisition.
2. Détecteur hybride → score, type d'attaque probable, carte.
3. Politique : **conforme** (< τ₁, choisi pour 1 % de faux positifs sur la validation), **à vérifier** (τ₁–τ₂), **alerte** (> τ₂ ou empreinte discordante).
4. **Journal d'audit chaîné** (chaque entrée contient l'empreinte de la précédente).
5. Notification radiologue / RSSI.

In [ ]:
import pydicom, datetime, getpass
CRITICAL_TAGS = ["StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "RescaleSlope", "RescaleIntercept", "Rows", "Columns"]

def dicom_fingerprint(ds):
    h = hashlib.sha256(ds.PixelData)
    for t in CRITICAL_TAGS:
        h.update(str(getattr(ds, t, "")).encode())
    return h.hexdigest()

def dicom_to_hu(ds):
    return (ds.pixel_array.astype(np.float32) * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))).astype(np.int16)

class AuditLog:
    def __init__(self, path):
        self.path = Path(path); self.path.touch(exist_ok=True)
    def _last(self):
        lines = self.path.read_text().strip().splitlines()
        return json.loads(lines[-1])["entry_hash"] if lines else "0" * 64
    def append(self, record):
        rec = dict(record, prev_hash=self._last(), time=datetime.datetime.now().isoformat(timespec="seconds"))
        rec["entry_hash"] = hashlib.sha256(json.dumps(rec, sort_keys=True).encode()).hexdigest()
        with self.path.open("a") as f: f.write(json.dumps(rec) + "\n")
        return rec
    def verify(self):
        prev = "0" * 64
        for i, line in enumerate(self.path.read_text().strip().splitlines()):
            rec = json.loads(line); h = rec.pop("entry_hash")
            if rec["prev_hash"] != prev or hashlib.sha256(json.dumps(rec, sort_keys=True).encode()).hexdigest() != h:
                return False, i
            prev = h
        return True, None

# Seuils calibrés sur la validation : τ1 -> 1 % de FP, τ2 -> 0,1 % de FP (ou 0,9 si trop peu de données)
val_auth = np.array([slice_scores(BO[k])["hybride"] for k in vkeys if DATA[k]["attack"] == "authentique"])
TAU1 = float(np.quantile(val_auth, 0.99)); TAU2 = float(max(np.quantile(val_auth, 0.999), min(0.9, TAU1 + 0.2)))
AUDIT = AuditLog(OUT / "audit_log.jsonl")
REGISTRY = {}   # empreintes enregistrées à l'acquisition (SOPInstanceUID -> sha256), à brancher sur la base de l'établissement

def verify_dicom(path, user="service-verif"):
    ds = pydicom.dcmread(path)
    fp = dicom_fingerprint(ds); ref = REGISTRY.get(str(getattr(ds, "SOPInstanceUID", "")))
    img = dicom_to_hu(ds); lung = lung_mask_slice(img)
    bo = branch_outputs(img, lung)
    score = slice_scores(bo)["hybride"] if bo else 0.0
    pmap = pixel_map(bo["coords"], fused_local(bo), img.shape) if bo else np.zeros(img.shape)
    hash_ok = (ref is None) or (ref == fp)
    verdict = "ALERTE" if (not hash_ok or score > TAU2) else ("A VERIFIER" if score > TAU1 else "CONFORME")
    rec = AUDIT.append(dict(sop=str(getattr(ds, "SOPInstanceUID", "")), sha256=fp, hash_ok=hash_ok, score=round(float(score), 4),
                            verdict=verdict, user=user, tau1=round(TAU1, 4), tau2=round(TAU2, 4)))
    return verdict, score, img, pmap, rec

print("τ1 =", round(TAU1, 3), "| τ2 =", round(TAU2, 3)); print("Journal intègre :", AUDIT.verify())

In [ ]:
# Interface de démonstration (Gradio) — à exécuter sur un poste du réseau interne uniquement
try:
    import gradio as gr
except ImportError:
    gr = None
    print("Gradio non installé : interface de démonstration ignorée (pip install gradio).")
def ui(file):
    verdict, score, img, pmap, rec = verify_dicom(file.name)
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    w = np.clip((img + 1350) / 1500, 0, 1)
    ax[0].imshow(w, cmap="gray"); ax[0].set_title("Coupe"); ax[1].imshow(w, cmap="gray")
    ax[1].imshow(np.ma.masked_less(pmap, 0.05), cmap="magma", vmin=0, vmax=1, alpha=0.7); ax[1].set_title("Carte de falsification")
    for a in ax: a.axis("off")
    return f"{verdict} (score {score:.3f})", fig, json.dumps(rec, indent=1)
if gr is not None:
    demo = gr.Interface(fn=ui, inputs=gr.File(label="Fichier DICOM"), outputs=[gr.Textbox(label="Verdict"), gr.Plot(), gr.Code(label="Entrée du journal d'audit")],
                        title="Vérification d'authenticité des images CT")
# demo.launch(share=False)

**Intégration PACS (Orthanc).** Orthanc déclenche un script Python à chaque instance reçue (`OnStoredInstance`) ; le script récupère le fichier via `GET /instances/{id}/file`, appelle `verify_dicom`, puis écrit le verdict dans les métadonnées de l'instance (`PUT /instances/{id}/metadata/...`) et notifie en cas d'alerte. Les analyses restent **sur site** : aucune image ne quitte l'établissement (loi n° 2024/017).

```python
# greffon Python d'Orthanc (extrait)
import orthanc, io
def OnChange(changeType, level, resource):
    if changeType == orthanc.ChangeType.NEW_INSTANCE:
        dcm = orthanc.RestApiGet(f"/instances/{resource}/file")
        verdict, score, *_ = verify_dicom(io.BytesIO(dcm))
        orthanc.RestApiPut(f"/instances/{resource}/metadata/1024", verdict.encode())
orthanc.RegisterOnChangeCallback(OnChange)
```

## 11. Preuve de concept SimCT-Tamper (reproduction des chiffres du chapitre 3)

Ces cellules écrivent puis exécutent les scripts de la preuve de concept (sans GPU, ~10 min sur 2 cœurs). `features.py` a été écrit en §6.

In [ ]:
%%writefile simct.py
"""Simulateur de coupes CT thoraciques + générateur de falsifications (preuve de concept).

Chaîne physique : carte HU -> coefficients d'atténuation -> projection de Radon ->
bruit de Poisson (dose I0) -> rétroprojection filtrée (FBP). Le bruit obtenu est donc
corrélé spatialement (texture « CT » réelle), ce qui est la base des indices forensiques.
"""
import numpy as np
import cv2
from scipy import ndimage as ndi
from skimage.transform import radon, iradon
import warnings

warnings.filterwarnings("ignore")
N = 256
MU_W = 0.02  # mm^-1 approx (eau) ; pixel = 1.4 mm
PIX = 1.4
THETA = np.linspace(0, 180, 360, endpoint=False)


def ellipse(shape, cy, cx, ry, rx, ang=0.0):
    yy, xx = np.mgrid[: shape[0], : shape[1]]
    y, x = yy - cy, xx - cx
    c, s = np.cos(ang), np.sin(ang)
    xr, yr = c * x + s * y, -s * x + c * y
    return (xr / rx) ** 2 + (yr / ry) ** 2 <= 1.0


def nodule_profile(r, rng, spic=True):
    """Nodule pulmonaire 2D : profil lisse + spiculations éventuelles (en HU relatifs)."""
    size = int(2 * r + 9)
    yy, xx = np.mgrid[:size, :size] - size // 2
    rho = np.sqrt(yy ** 2 + xx ** 2)
    ang = np.arctan2(yy, xx)
    rr = r * (1 + (0.15 * np.sin(rng.integers(3, 8) * ang + rng.uniform(0, 6)) if spic else 0))
    prof = 1.0 / (1 + np.exp((rho - rr) / 0.8))
    return prof  # 0..1


def make_phantom(rng, with_nodule=False):
    """Retourne une carte HU (N x N) et la liste des nodules (cy, cx, r)."""
    hu = np.full((N, N), -1000.0)
    by, bx = N / 2 + rng.normal(0, 3), N / 2 + rng.normal(0, 3)
    ry, rx = rng.uniform(78, 90), rng.uniform(105, 118)
    body = ellipse(hu.shape, by, bx, ry, rx)
    hu[body] = -90 + rng.normal(0, 5)  # graisse sous-cutanée
    muscle = ellipse(hu.shape, by, bx, ry - 8, rx - 8)
    hu[muscle] = 45 + rng.normal(0, 5)
    # poumons
    lungs = np.zeros_like(body)
    for side in (-1, 1):
        lungs |= ellipse(hu.shape, by - 4 + rng.normal(0, 2), bx + side * rng.uniform(44, 52),
                         rng.uniform(55, 66), rng.uniform(30, 38), ang=side * rng.uniform(0.05, 0.2))
    tex = ndi.gaussian_filter(rng.normal(0, 1, hu.shape), 2.0)
    hu[lungs] = -860 + 25 * tex[lungs]
    # vaisseaux : segments aléatoires lissés
    ves = np.zeros(hu.shape, np.float32)
    for _ in range(rng.integers(25, 45)):
        p = (int(rng.uniform(40, 216)), int(rng.uniform(40, 216)))
        q = (p[0] + int(rng.normal(0, 18)), p[1] + int(rng.normal(0, 18)))
        cv2.line(ves, p, q, 1.0, int(rng.integers(1, 3)))
    ves = ndi.gaussian_filter(ves, 0.8) * lungs
    hu += ves * rng.uniform(600, 850)
    # coeur, aorte, vertèbre, côtes
    hu[ellipse(hu.shape, by + 12, bx + 8, 30, 34)] = 40 + rng.normal(0, 4)
    hu[ellipse(hu.shape, by - 20, bx + 12, 9, 9)] = 45
    hu[ellipse(hu.shape, by + ry - 26, bx, 13, 12)] = 700 + rng.normal(0, 30)
    hu[ellipse(hu.shape, by + ry - 26, bx, 8, 7)] = 250
    for k in range(8):
        a = np.pi * (0.15 + 0.7 * k / 7)
        for side in (-1, 1):
            cy = by - (ry - 6) * np.cos(a) * 0.95
            cx = bx + side * (rx - 6) * np.sin(a)
            hu[ellipse(hu.shape, cy, cx, 3.5, 5)] = 650
    nods = []
    if with_nodule:
        lung_idx = np.argwhere(ndi.binary_erosion(lungs, iterations=12))
        for _ in range(rng.integers(1, 3)):
            cy, cx = lung_idx[rng.integers(len(lung_idx))]
            r = rng.uniform(3, 8)
            prof = nodule_profile(r, rng)
            h = prof.shape[0] // 2
            sl = (slice(cy - h, cy - h + prof.shape[0]), slice(cx - h, cx - h + prof.shape[1]))
            hu[sl] = hu[sl] * (1 - prof) + (rng.uniform(10, 60)) * prof
            nods.append((int(cy), int(cx), float(r)))
    return hu, lungs, nods


def acquire(hu, rng, I0, kernel):
    """Acquisition CT simulée : Radon + bruit de Poisson + FBP. Retourne une image en HU (int16)."""
    mu = np.clip(MU_W * (1 + hu / 1000.0), 0, None) * PIX
    circ = ellipse(hu.shape, N / 2 - 0.5, N / 2 - 0.5, N / 2 - 1, N / 2 - 1)
    mu = mu * circ
    sino = radon(mu, THETA, circle=True)
    counts = rng.poisson(I0 * np.exp(-sino)).astype(np.float64)
    counts = np.maximum(counts, 1)
    sino_n = -np.log(counts / I0)
    rec = iradon(sino_n, THETA, circle=True, filter_name=kernel) / PIX
    out = (rec / MU_W - 1) * 1000.0
    out[~circ] = -1024
    return np.clip(np.round(out), -1024, 3071).astype(np.int16)


# ----------------------------------------------------------------------------------
# Estimation de bruit et falsifications (côté attaquant)
# ----------------------------------------------------------------------------------

def mad_sigma(img, mask):
    r = img.astype(np.float32) - ndi.median_filter(img.astype(np.float32), 3)
    v = r[mask]
    return 1.4826 * np.median(np.abs(v - np.median(v))) * 1.2


def blend_patch(img, patch, cy, cx, rng, sigma_noise):
    """Colle un patch (HU float) avec fenêtre gaussienne + bruit blanc ajusté (camouflage)."""
    out = img.astype(np.float32).copy()
    h, w = patch.shape
    y0, x0 = cy - h // 2, cx - w // 2
    yy, xx = np.mgrid[:h, :w]
    win = np.exp(-(((yy - h / 2) / (h / 3.2)) ** 2 + ((xx - w / 2) / (w / 3.2)) ** 2))
    region = out[y0:y0 + h, x0:x0 + w]
    noisy = patch + rng.normal(0, sigma_noise, patch.shape)
    out[y0:y0 + h, x0:x0 + w] = region * (1 - win) + noisy * win
    mask = np.zeros(img.shape, bool)
    mask[y0:y0 + h, x0:x0 + w] = win > 0.25
    return np.clip(np.round(out), -1024, 3071).astype(np.int16), mask


def tamper_injection(img, lungs, rng):
    idx = np.argwhere(ndi.binary_erosion(lungs, iterations=16))
    cy, cx = idx[rng.integers(len(idx))]
    r = rng.uniform(5, 9)
    prof = nodule_profile(r, rng)
    pad = 6
    prof = np.pad(prof, pad)
    h = prof.shape[0]
    base = ndi.gaussian_filter(img.astype(np.float32), 1.5)[cy - h // 2: cy - h // 2 + h, cx - h // 2: cx - h // 2 + h]
    patch = base * (1 - prof) + rng.uniform(10, 50) * prof
    sig = mad_sigma(img, lungs)
    return blend_patch(img, patch, cy, cx, rng, sig)


def tamper_removal(img, lungs, nods, rng):
    cy, cx, r = nods[0]
    h = int(2 * r + 14)
    m = np.zeros(img.shape, np.uint8)
    cv2.circle(m, (cx, cy), int(r + 3), 1, -1)
    win = ((img.astype(np.float32) + 1024) / 4095 * 255).astype(np.uint8)
    # inpainting en HU via image 16 bits normalisée -> on inpaint sur float32 via cv2 (8 bits) puis reprojection
    f = img.astype(np.float32)
    lo, hi = -1024.0, 400.0
    u8 = np.clip((f - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)
    inp = cv2.inpaint(u8, m, 5, cv2.INPAINT_TELEA).astype(np.float32) / 255 * (hi - lo) + lo
    patch = ndi.gaussian_filter(inp, 1.0)[cy - h // 2: cy - h // 2 + h, cx - h // 2: cx - h // 2 + h]
    sig = mad_sigma(img, lungs)
    return blend_patch(img, patch, cy, cx, rng, sig)


def tamper_copymove(img, lungs, rng):
    idx = np.argwhere(ndi.binary_erosion(lungs, iterations=20))
    s = int(rng.integers(20, 30))
    for _ in range(50):
        a = idx[rng.integers(len(idx))]
        b = idx[rng.integers(len(idx))]
        if np.abs(a - b).max() > s + 4:
            break
    out = img.copy()
    src = img[a[0] - s // 2: a[0] - s // 2 + s, a[1] - s // 2: a[1] - s // 2 + s]
    out[b[0] - s // 2: b[0] - s // 2 + s, b[1] - s // 2: b[1] - s // 2 + s] = src
    mask = np.zeros(img.shape, bool)
    mask[b[0] - s // 2: b[0] - s // 2 + s, b[1] - s // 2: b[1] - s // 2 + s] = True
    return out, mask


def tamper_noise(img, rng):
    """Injection de bruit globale (perturbation de type adversariale non ciblée)."""
    sig = rng.uniform(4, 12)
    n = rng.normal(0, sig, img.shape)
    body = img > -900
    out = img.astype(np.float32) + n * body
    return np.clip(np.round(out), -1024, 3071).astype(np.int16), body


class PPCAGenerator:
    """Modèle génératif linéaire-gaussien (PCA probabiliste) appris sur des patchs nodulaires authentiques.
    Sert de *proxy* aux générateurs profonds (GAN/diffusion) : il produit des contenus plausibles mais
    dont la signature de bruit n'est pas celle de la chaîne d'acquisition."""

    def __init__(self, k=24):
        self.k = k

    def fit(self, patches):
        X = patches.reshape(len(patches), -1).astype(np.float64)
        self.mu = X.mean(0)
        U, S, Vt = np.linalg.svd(X - self.mu, full_matrices=False)
        self.W = Vt[: self.k]
        self.std = S[: self.k] / np.sqrt(len(X) - 1)
        self.shape = patches.shape[1:]
        return self

    def sample(self, rng):
        z = rng.normal(0, 1, self.k) * self.std
        return (self.mu + z @ self.W).reshape(self.shape)


def tamper_generative(img, lungs, gen, rng):
    idx = np.argwhere(ndi.binary_erosion(lungs, iterations=18))
    cy, cx = idx[rng.integers(len(idx))]
    patch = gen.sample(rng)
    sig = mad_sigma(img, lungs)
    return blend_patch(img, patch, cy, cx, rng, sig)

In [ ]:
%%writefile generate.py
"""Génère le jeu de données de la preuve de concept (SimCT-Tamper)."""
import numpy as np
from multiprocessing import Pool
import simct as S

N_PAT, N_AUTH, N_PER_ATT = 36, 15, 3
N_GEN_PAT = 6
KERNELS = ["ramp", "shepp-logan", "cosine", "hann"]
CLASSES = ["authentique", "injection", "suppression", "copier-deplacer", "bruit", "generatif"]


def scanner(pid):
    r = np.random.default_rng(1000 + pid)
    return float(r.uniform(2e4, 1.2e5)), KERNELS[pid % len(KERNELS)]


def acquire_one(args):
    pid, j, with_nod = args
    rng = np.random.default_rng(pid * 1000 + j)
    I0, ker = scanner(pid)
    hu, lungs, nods = S.make_phantom(rng, with_nodule=with_nod)
    img = S.acquire(hu, rng, I0, ker)
    return pid, j, img, lungs, nods


def main():
    # 1) Patients « attaquant » pour apprendre le générateur PPCA
    jobs = [(100 + p, j, True) for p in range(N_GEN_PAT) for j in range(12)]
    # 2) Patients de l'étude
    plan = []
    for p in range(N_PAT):
        labels = [0] * N_AUTH + sum([[c] * N_PER_ATT for c in range(1, 6)], [])
        for j, c in enumerate(labels):
            rng = np.random.default_rng(p * 77 + j)
            with_nod = True if c == 2 else (rng.random() < 0.5)
            plan.append((p, j, c, with_nod))
    jobs += [(p, j, wn) for p, j, c, wn in plan]
    with Pool(2) as pool:
        res = pool.map(acquire_one, jobs, chunksize=8)
    gen_res = [r for r in res if r[0] >= 100]
    stu = {(r[0], r[1]): r for r in res if r[0] < 100}

    # Générateur
    patches = []
    for _, _, img, lungs, nods in gen_res:
        for cy, cx, r in nods:
            h = 14
            pt = img[cy - h:cy + h, cx - h:cx + h].astype(np.float32)
            if pt.shape == (28, 28):
                patches.append(pt)
                patches.append(pt[:, ::-1]); patches.append(pt[::-1]); patches.append(pt.T)
    gen = S.PPCAGenerator(k=24).fit(np.array(patches))
    print("patchs PPCA:", len(patches))

    X, M, Y, P, L = [], [], [], [], []
    for p, j, c, _ in plan:
        _, _, img, lungs, nods = stu[(p, j)]
        rng = np.random.default_rng(10_000 + p * 100 + j)
        if c == 0:
            out, m = img, np.zeros(img.shape, bool)
        elif c == 1:
            out, m = S.tamper_injection(img, lungs, rng)
        elif c == 2:
            out, m = S.tamper_removal(img, lungs, nods, rng)
        elif c == 3:
            out, m = S.tamper_copymove(img, lungs, rng)
        elif c == 4:
            out, m = S.tamper_noise(img, rng)
        else:
            out, m = S.tamper_generative(img, lungs, gen, rng)
        X.append(out); M.append(m); Y.append(c); P.append(p); L.append(lungs)
    np.savez_compressed("simct_tamper.npz", X=np.array(X), M=np.packbits(np.array(M), axis=-1),
                        Y=np.array(Y), P=np.array(P), L=np.packbits(np.array(L), axis=-1))
    print("ok", len(X), np.bincount(Y))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile experiments.py
"""Expériences de la preuve de concept : détection, typage, généralisation, robustesse, localisation."""
import json, time
import numpy as np
from multiprocessing import Pool
from scipy import ndimage as ndi
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, roc_curve)
import features as F

CLASSES = ["authentique", "injection", "suppression", "copier-déplacer", "bruit", "génératif"]
d = np.load("simct_tamper.npz")
X = d["X"]; Y = d["Y"]; P = d["P"]
M = np.unpackbits(d["M"], axis=-1)[:, :, :256].astype(bool)
L = np.unpackbits(d["L"], axis=-1)[:, :, :256].astype(bool)
yb = (Y > 0).astype(int)


def feat_job(i):
    t = time.perf_counter()
    f, (blocks, bf, dup) = F.image_features(X[i], L[i])
    dt = time.perf_counter() - t
    return f, F.intensity_features(X[i], L[i]), blocks, bf, dup, dt


def feat_img(img, lungs):
    return F.image_features(img, lungs)[0]


if __name__ == "__main__":
    with Pool(2) as pool:
        res = pool.map(feat_job, range(len(X)), chunksize=10)
    FX = np.nan_to_num(np.array([r[0] for r in res]))
    IX = np.nan_to_num(np.array([r[1] for r in res]))
    times = np.array([r[5] for r in res])
    out = {"n": int(len(X)), "class_counts": np.bincount(Y).tolist(), "n_feat": int(FX.shape[1]),
           "feat_time_ms_mean": float(times.mean() * 1000), "feat_time_ms_std": float(times.std() * 1000)}

    models = {
        "Régression logistique": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0)),
        "Random Forest": lambda: RandomForestClassifier(400, n_jobs=2, random_state=0),
        "Gradient Boosting (HGB)": lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0),
        "Perceptron multicouche": lambda: make_pipeline(StandardScaler(), MLPClassifier((128, 64), max_iter=2000, alpha=1e-3, random_state=0)),
    }
    gkf = GroupKFold(5)

    def cv_eval(Xf, mk, y=yb):
        rows, prob = [], np.zeros(len(y))
        for tr, te in gkf.split(Xf, y, P):
            m = mk(); m.fit(Xf[tr], y[tr])
            p = m.predict_proba(Xf[te])[:, 1]; prob[te] = p
            pr = (p >= 0.5).astype(int)
            rows.append([accuracy_score(y[te], pr), precision_score(y[te], pr, zero_division=0),
                         recall_score(y[te], pr), f1_score(y[te], pr), roc_auc_score(y[te], p)])
        r = np.array(rows)
        return {"mean": r.mean(0).tolist(), "std": r.std(0).tolist()}, prob

    E1 = {}
    probs = {}
    E1["Ligne de base intensité (RF)"], probs["base"] = cv_eval(IX, models["Random Forest"])
    for name, mk in models.items():
        E1[name], probs[name] = cv_eval(FX, mk)
    E1["Forensique + intensité (HGB)"], _ = cv_eval(np.hstack([FX, IX]), models["Gradient Boosting (HGB)"])
    out["E1"] = E1
    best = max([k for k in models], key=lambda k: E1[k]["mean"][4])
    out["best"] = best
    pb = probs[best]
    out["per_class_detection"] = {CLASSES[c]: float(((pb >= 0.5) == (c > 0))[Y == c].mean()) for c in range(6)}
    fpr, tpr, _ = roc_curve(yb, pb); fb, tb, _ = roc_curve(yb, probs["base"])
    np.savez("roc.npz", fpr=fpr, tpr=tpr, fb=fb, tb=tb)
    out["tpr_at_fpr1"] = float(np.interp(0.01, fpr, tpr)); out["tpr_at_fpr5"] = float(np.interp(0.05, fpr, tpr))

    # Ablation par groupes de caractéristiques
    groups = {"Statistiques globales des résidus": list(range(0, 12)),
              "Autocorrélation + spectre du bruit": list(range(12, 20)),
              "Incohérence locale (blocs)": list(range(20, 38)),
              "Duplication de blocs": list(range(38, 40))}
    ab = {}
    for g, idx in groups.items():
        ab[g], _ = cv_eval(FX[:, idx], models["Gradient Boosting (HGB)"])
    ab["Toutes"] = E1["Gradient Boosting (HGB)"]
    out["ablation"] = ab

    # E2 multiclasse
    pred = np.zeros(len(Y), int)
    for tr, te in gkf.split(FX, Y, P):
        m = models["Gradient Boosting (HGB)"](); m.fit(FX[tr], Y[tr]); pred[te] = m.predict(FX[te])
    out["E2"] = {"acc": float(accuracy_score(Y, pred)), "macroF1": float(f1_score(Y, pred, average="macro")),
                 "cm": confusion_matrix(Y, pred).tolist(),
                 "f1_per_class": f1_score(Y, pred, average=None).tolist()}

    # E3 leave-one-attack-out
    E3 = {}
    for c in range(1, 6):
        aucs = []
        for tr, te in gkf.split(FX, Y, P):
            trm = tr[Y[tr] != c]
            tem = te[(Y[te] == 0) | (Y[te] == c)]
            m = models["Gradient Boosting (HGB)"](); m.fit(FX[trm], yb[trm])
            aucs.append(roc_auc_score(yb[tem], m.predict_proba(FX[tem])[:, 1]))
        E3[CLASSES[c]] = [float(np.mean(aucs)), float(np.std(aucs))]
    out["E3"] = E3

    # E4 robustesse aux post-traitements (split 70/30 par patient)
    tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=1).split(FX, yb, P))
    m = models["Gradient Boosting (HGB)"](); m.fit(FX[tr], yb[tr])
    rng = np.random.default_rng(5)

    def pp(img, kind):
        f = img.astype(np.float32)
        if kind == "Flou gaussien σ=0,5":
            f = ndi.gaussian_filter(f, 0.5)
        elif kind == "Redimensionnement ×0,75":
            f = ndi.zoom(ndi.zoom(f, 0.75, order=1), 256 / 192, order=1)[:256, :256]
        elif kind == "Bruit additif σ=2 HU":
            f = f + rng.normal(0, 2, f.shape)
        elif kind == "Export 8 bits (fenêtre pulmonaire)":
            lo, hi = -1350, 150
            f = np.round(np.clip((f - lo) / (hi - lo), 0, 1) * 255) / 255 * (hi - lo) + lo
        return np.clip(np.round(f), -1024, 3071).astype(np.int16)

    E4 = {"Aucun": float(roc_auc_score(yb[te], m.predict_proba(FX[te])[:, 1]))}
    kinds = ["Flou gaussien σ=0,5", "Redimensionnement ×0,75", "Bruit additif σ=2 HU", "Export 8 bits (fenêtre pulmonaire)"]
    with Pool(2) as pool:
        for k in kinds:
            Fk = np.nan_to_num(np.array(pool.starmap(feat_img, [(pp(X[i], k), L[i]) for i in te])))
            E4[k] = float(roc_auc_score(yb[te], m.predict_proba(Fk)[:, 1]))
            # ré-entraînement avec augmentation (mêmes post-traitements appliqués à 50 % du train)
    out["E4"] = E4

    # E5 localisation par blocs (attaques locales)
    loc_cls = [1, 2, 3, 5]
    Bk = []
    for i, r in enumerate(res):
        blocks, bf, dup = r[2], r[3], r[4]
        if len(blocks) == 0:
            continue
        z = F.robust_z(bf)
        for (y, x), f, zz in zip(blocks, bf, z):
            lab = M[i, y:y + 16, x:x + 16].mean() > 0.3 if Y[i] in loc_cls else False
            Bk.append([i, y, x, *f, *np.abs(zz), dup[y:y + 16, x:x + 16].mean(), lab])
    Bk = np.nan_to_num(np.array(Bk, np.float32))
    bi = Bk[:, 0].astype(int)
    keep = np.isin(Y[bi], [0] + loc_cls)
    Bk, bi = Bk[keep], bi[keep]
    bX, by = Bk[:, 3:-1], Bk[:, -1].astype(int)
    bprob = np.zeros(len(by)); uns = np.abs(Bk[:, 9:15]).max(1) + 5 * Bk[:, 15]
    for trp, tep in gkf.split(bX, by, P[bi]):
        mm = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0, class_weight="balanced")
        mm.fit(bX[trp], by[trp]); bprob[tep] = mm.predict_proba(bX[tep])[:, 1]
    out["E5_block_auc_supervised"] = float(roc_auc_score(by, bprob))
    out["E5_block_auc_unsupervised"] = float(roc_auc_score(by, uns))

    def pixmap(i, score):
        sel = bi == i
        acc = np.zeros((256, 256)); cnt = np.zeros((256, 256))
        for (y, x), s in zip(Bk[sel, 1:3].astype(int), score[sel]):
            acc[y:y + 16, x:x + 16] += s; cnt[y:y + 16, x:x + 16] += 1
        return np.where(cnt > 0, acc / np.maximum(cnt, 1), 0)

    dice = {c: [] for c in loc_cls}; iou = {c: [] for c in loc_cls}
    maps = {}
    for i in np.where(np.isin(Y, loc_cls))[0]:
        pm = pixmap(i, bprob)
        pr = pm >= 0.5
        gt = M[i] & L[i]
        if gt.sum() == 0:
            gt = M[i]
        inter = (pr & gt).sum(); uni = (pr | gt).sum()
        dice[Y[i]].append(2 * inter / (pr.sum() + gt.sum() + 1e-9)); iou[Y[i]].append(inter / (uni + 1e-9))
        if Y[i] not in maps:
            maps[int(Y[i])] = (int(i), pm)
    out["E5_dice"] = {CLASSES[c]: [float(np.mean(v)), float(np.std(v))] for c, v in dice.items()}
    out["E5_iou"] = {CLASSES[c]: [float(np.mean(v)), float(np.std(v))] for c, v in iou.items()}
    np.savez("maps.npz", **{f"m{c}": v[1] for c, v in maps.items()}, idx=np.array([maps[c][0] for c in sorted(maps)]),
             cls=np.array(sorted(maps)))
    # importance des caractéristiques (permutation simplifiée via HGB n'expose pas) -> RF
    rf = RandomForestClassifier(400, n_jobs=2, random_state=0).fit(FX, yb)
    out["rf_importance"] = rf.feature_importances_.tolist()
    np.save("probs_best.npy", pb)
    json.dump(out, open("results.json", "w"), ensure_ascii=False, indent=1)
    print(json.dumps({k: v for k, v in out.items() if k not in ("rf_importance",)}, ensure_ascii=False, indent=1))

In [ ]:
%%writefile hybrid.py
"""Approche proposée (PoC) : détection hiérarchique local -> global.
Étape 1 : classifieur de blocs (probabilité de falsification locale).
Étape 2 : agrégation (max, moyenne top-k) + descripteurs globaux -> décision image.
Validation croisée imbriquée groupée par patient (aucune fuite entre patients)."""
import json, time
import numpy as np
from multiprocessing import Pool
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve,
                             confusion_matrix)
import features as F

CLASSES = ["authentique", "injection", "suppression", "copier-déplacer", "bruit", "génératif"]
d = np.load("simct_tamper.npz")
X, Y, P = d["X"], d["Y"], d["P"]
M = np.unpackbits(d["M"], axis=-1)[:, :, :256].astype(bool)
L = np.unpackbits(d["L"], axis=-1)[:, :, :256].astype(bool)
yb = (Y > 0).astype(int)


def job(i):
    f, (blocks, bf, dup) = F.image_features(X[i], L[i])
    rows = []
    if len(blocks):
        z = np.abs(F.robust_z(bf))
        img = X[i].astype(np.float32)
        for (y, x), b, zz in zip(blocks, bf, z):
            lab = (M[i, y:y + 16, x:x + 16].mean() > 0.3) and Y[i] in (1, 2, 3, 5)
            blk = img[y:y + 16, x:x + 16]
            rows.append([i, y, x, *b, *zz, dup[y:y + 16, x:x + 16].mean(), blk.mean(), blk.std(), lab])
    return f, np.array(rows, np.float32)


def hgb():
    return HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, random_state=0)


def agg(bi, prob, n):
    out = np.zeros((n, 4))
    for i in range(n):
        p = np.sort(prob[bi == i])[::-1]
        if len(p) == 0:
            continue
        out[i] = [p[0], p[:3].mean(), p[:10].mean(), (p > 0.5).sum()]
    return out


if __name__ == "__main__":
    t0 = time.time()
    with Pool(2) as pool:
        res = pool.map(job, range(len(X)), chunksize=10)
    FX = np.nan_to_num(np.array([r[0] for r in res]))
    BK = np.nan_to_num(np.vstack([r[1] for r in res if len(r[1])]))
    np.savez("cache.npz", FX=FX, BK=BK)
    bi, bX, by = BK[:, 0].astype(int), BK[:, 3:-1], BK[:, -1].astype(int)
    pid = np.unique(P)
    fold_of = {p: k % 5 for k, p in enumerate(np.random.default_rng(0).permutation(pid))}
    fold = np.array([fold_of[p] for p in P])
    n = len(X)
    prob_img = np.zeros(n); rows = []; block_prob_test = np.zeros(len(by))
    for k in range(5):
        te_i = np.where(fold == k)[0]; tr_i = np.where(fold != k)[0]
        te_b = np.isin(bi, te_i); tr_b = np.isin(bi, tr_i)
        # méta-caractéristiques OOF internes pour le train
        inner = np.zeros(len(by))
        for kk in [j for j in range(5) if j != k]:
            itr = tr_b & ~np.isin(bi, np.where(fold == kk)[0]); ite = tr_b & np.isin(bi, np.where(fold == kk)[0])
            m = hgb().fit(bX[itr], by[itr]); inner[ite] = m.predict_proba(bX[ite])[:, 1]
        mb = hgb().fit(bX[tr_b], by[tr_b])
        block_prob_test[te_b] = mb.predict_proba(bX[te_b])[:, 1]
        A_tr = agg(bi, inner, n)[tr_i]; A_te = agg(bi, block_prob_test, n)[te_i]
        mi = hgb().fit(np.hstack([FX[tr_i], A_tr]), yb[tr_i])
        p = mi.predict_proba(np.hstack([FX[te_i], A_te]))[:, 1]; prob_img[te_i] = p
        pr = (p >= 0.5).astype(int)
        rows.append([accuracy_score(yb[te_i], pr), precision_score(yb[te_i], pr), recall_score(yb[te_i], pr),
                     f1_score(yb[te_i], pr), roc_auc_score(yb[te_i], p)])
    r = np.array(rows)
    out = {"hybrid": {"mean": r.mean(0).tolist(), "std": r.std(0).tolist()}}
    out["hybrid_per_class"] = {CLASSES[c]: float(((prob_img >= 0.5) == (c > 0))[Y == c].mean()) for c in range(6)}
    fpr, tpr, _ = roc_curve(yb, prob_img)
    out["tpr_at_fpr1"] = float(np.interp(0.01, fpr, tpr)); out["tpr_at_fpr5"] = float(np.interp(0.05, fpr, tpr))
    out["block_auc_nested"] = float(roc_auc_score(by, block_prob_test))
    # localisation à partir des probabilités de blocs (test externe)
    dice, iou, maps = {}, {}, {}
    for c in (1, 2, 3, 5):
        dl, il = [], []
        for i in np.where(Y == c)[0]:
            sel = bi == i
            acc = np.zeros((256, 256)); cnt = np.zeros((256, 256))
            for (y, x), s in zip(BK[sel, 1:3].astype(int), block_prob_test[sel]):
                acc[y:y + 16, x:x + 16] += s; cnt[y:y + 16, x:x + 16] += 1
            pm = np.where(cnt > 0, acc / np.maximum(cnt, 1), 0)
            pr = pm >= 0.5; gt = M[i] & L[i]
            inter = (pr & gt).sum()
            dl.append(2 * inter / (pr.sum() + gt.sum() + 1e-9)); il.append(inter / ((pr | gt).sum() + 1e-9))
            if c not in maps or dl[-1] > 0.5 and maps[c][2] < 0.5:
                maps[c] = (i, pm, dl[-1])
        dice[CLASSES[c]] = [float(np.mean(dl)), float(np.std(dl)), float(np.median(dl))]
        iou[CLASSES[c]] = [float(np.mean(il)), float(np.std(il))]
    out["dice"] = dice; out["iou"] = iou
    # comptage détection locale : au moins un bloc correct > 0.5
    out["time_s"] = time.time() - t0
    fprs, tprs, _ = roc_curve(yb, prob_img)
    np.savez("roc_hybrid.npz", fpr=fprs, tpr=tprs)
    np.savez("maps_hybrid.npz", idx=np.array([maps[c][0] for c in (1, 2, 3, 5)]),
             **{f"m{c}": maps[c][1] for c in (1, 2, 3, 5)})
    np.save("prob_hybrid.npy", prob_img)
    json.dump(out, open("results_hybrid.json", "w"), ensure_ascii=False, indent=1)
    print(json.dumps(out, ensure_ascii=False, indent=1))

In [ ]:
%%writefile figures.py
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy import ndimage as ndi
import features as F

C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.edgecolor": INK2,
                     "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150})
OUT = "fig/"
import os; os.makedirs(OUT, exist_ok=True)
CL = ["Authentique", "Injection", "Suppression", "Copier-déplacer", "Bruit", "Génératif"]
d = np.load("simct_tamper.npz")
X, Y, P = d["X"], d["Y"], d["P"]
M = np.unpackbits(d["M"], axis=-1)[:, :, :256].astype(bool)
L = np.unpackbits(d["L"], axis=-1)[:, :, :256].astype(bool)
R = json.load(open("results.json")); H = json.load(open("results_hybrid.json"))


def win(img, lo=-1350, hi=150):
    return np.clip((img.astype(float) - lo) / (hi - lo), 0, 1)


def grid(ax):
    ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)


# Fig 1 : exemples de falsifications
fig, axs = plt.subplots(2, 6, figsize=(13, 4.6))
for c in range(6):
    i = np.where(Y == c)[0][3]
    axs[0, c].imshow(win(X[i]), cmap="gray"); axs[0, c].set_title(CL[c], fontsize=10)
    axs[1, c].imshow(win(X[i]), cmap="gray")
    if M[i].any():
        mm = ndi.binary_fill_holes(ndi.binary_closing(M[i], iterations=3)) if c == 4 else M[i]
        axs[1, c].contour(mm, levels=[0.5], colors=[C2], linewidths=1.2)
    for a in axs[:, c]:
        a.set_xticks([]); a.set_yticks([])
axs[0, 0].set_ylabel("Coupe"); axs[1, 0].set_ylabel("Vérité terrain")
plt.tight_layout(); plt.savefig(OUT + "fig_exemples.png", dpi=200); plt.close()

# Fig 2 : texture du bruit (résidu KV) authentique vs injection
i = np.where(Y == 1)[0][3]
ys, xs = np.where(M[i]); cy, cx = int(ys.mean()), int(xs.mean())
res = F.residuals(X[i])["kv"]
fig, axs = plt.subplots(1, 3, figsize=(10, 3.4))
axs[0].imshow(win(X[i][cy - 40:cy + 40, cx - 40:cx + 40]), cmap="gray"); axs[0].set_title("Zone injectée (fenêtre poumon)")
axs[1].imshow(np.clip(res[cy - 40:cy + 40, cx - 40:cx + 40], -60, 60), cmap="gray"); axs[1].set_title("Résidu haute fréquence (filtre KV)")
axs[1].contour(M[i][cy - 40:cy + 40, cx - 40:cx + 40], levels=[0.5], colors=[C2], linewidths=1)
# spectre radial résidu : bloc authentique vs bloc falsifié
def radial(r):
    Fm = np.abs(np.fft.fftshift(np.fft.fft2(r - r.mean()))) ** 2
    h = r.shape[0]; yy, xx = np.mgrid[:h, :h]; rad = np.sqrt((yy - h / 2) ** 2 + (xx - h / 2) ** 2).astype(int)
    return np.bincount(rad.ravel(), Fm.ravel()) / np.bincount(rad.ravel())
s_t = []; s_a = []
for j in np.where(Y == 1)[0][:60]:
    ys, xs = np.where(M[j]); y0, x0 = int(ys.mean()), int(xs.mean())
    r = F.residuals(X[j])["med"]
    s_t.append(radial(r[y0 - 8:y0 + 8, x0 - 8:x0 + 8])[:9])
    bl = F.lung_blocks(L[j])
    yb_, xb_ = bl[len(bl) // 3]
    s_a.append(radial(r[yb_:yb_ + 16, xb_:xb_ + 16])[:9])
f = np.arange(9) / 8
axs[2].plot(f, np.mean(s_a, 0) / np.mean(s_a, 0).sum(), color=C1, lw=2, marker="o", ms=5, label="Bloc authentique")
axs[2].plot(f, np.mean(s_t, 0) / np.mean(s_t, 0).sum(), color=C2, lw=2, marker="s", ms=5, label="Bloc falsifié")
axs[2].set_xlabel("Fréquence radiale normalisée"); axs[2].set_ylabel("Énergie relative")
axs[2].set_title("Spectre moyen du résidu (médian)"); axs[2].legend(frameon=False, fontsize=8); grid(axs[2])
for a in axs[:2]:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(OUT + "fig_bruit.png", dpi=200); plt.close()

# Fig 3 : ROC
r1 = np.load("roc.npz"); r2 = np.load("roc_hybrid.npz")
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.plot(r1["fb"], r1["tb"], color=C3, lw=2, label=f"Intensité seule (AUC = {R['E1']['Ligne de base intensité (RF)']['mean'][4]:.3f})")
ax.plot(r1["fpr"], r1["tpr"], color=C1, lw=2, label=f"Forensique globale (AUC = {R['E1'][R['best']]['mean'][4]:.3f})")
ax.plot(r2["fpr"], r2["tpr"], color=C2, lw=2, label=f"Hybride local→global (AUC = {H['hybrid']['mean'][4]:.3f})")
ax.plot([0, 1], [0, 1], color=INK2, lw=1, ls="--")
ax.set_xlabel("Taux de faux positifs"); ax.set_ylabel("Taux de vrais positifs"); ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(OUT + "fig_roc.png", dpi=200); plt.close()

# Fig 4 : détection par type d'attaque
pc_g = [R["per_class_detection"][k] for k in ["authentique", "injection", "suppression", "copier-déplacer", "bruit", "génératif"]]
pc_h = [H["hybrid_per_class"][k] for k in ["authentique", "injection", "suppression", "copier-déplacer", "bruit", "génératif"]]
fig, ax = plt.subplots(figsize=(8, 3.8)); x = np.arange(6); w = 0.38
b1 = ax.bar(x - w / 2 - 0.01, pc_g, w, color=C1, label="Forensique globale")
b2 = ax.bar(x + w / 2 + 0.01, pc_h, w, color=C2, label="Hybride local→global")
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.015, f"{b.get_height():.2f}", ha="center", fontsize=7, color=INK2)
ax.set_xticks(x); ax.set_xticklabels(CL); ax.set_ylim(0, 1.1); ax.set_ylabel("Taux de décision correcte")
ax.legend(frameon=False, ncol=2, loc="upper left"); grid(ax)
plt.tight_layout(); plt.savefig(OUT + "fig_par_classe.png", dpi=200); plt.close()

# Fig 5 : matrice de confusion
cm = np.array(R["E2"]["cm"]); cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for a in range(6):
    for b in range(6):
        ax.text(b, a, f"{cm[a, b]}", ha="center", va="center", fontsize=8, color="white" if cmn[a, b] > 0.55 else INK)
ax.set_xticks(range(6)); ax.set_xticklabels(CL, rotation=35, ha="right"); ax.set_yticks(range(6)); ax.set_yticklabels(CL)
ax.set_xlabel("Classe prédite"); ax.set_ylabel("Classe réelle"); ax.spines[:].set_visible(False)
plt.colorbar(im, fraction=0.046, label="Proportion par ligne")
plt.tight_layout(); plt.savefig(OUT + "fig_confusion.png", dpi=200); plt.close()

# Fig 6 : ablation
ab = R["ablation"]; names = list(ab.keys()); vals = [ab[k]["mean"][4] for k in names]; sd = [ab[k]["std"][4] for k in names]
fig, ax = plt.subplots(figsize=(7.5, 3.4))
yy = np.arange(len(names))
ax.barh(yy, vals, xerr=sd, color=C1, height=0.55, error_kw=dict(ecolor=INK2, lw=1, capsize=3))
for k, v in enumerate(vals):
    ax.text(v + 0.03, k, f"{v:.3f}", va="center", fontsize=8, color=INK2)
ax.set_yticks(yy); ax.set_yticklabels(names); ax.set_xlim(0.4, 0.85); ax.set_xlabel("AUC (validation croisée groupée, 5 plis)")
ax.axvline(0.5, color=INK2, lw=1, ls="--"); ax.invert_yaxis(); ax.grid(axis="x", color=GRID); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(OUT + "fig_ablation.png", dpi=200); plt.close()

# Fig 7 : robustesse + généralisation
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
e4 = R["E4"]; k4 = list(e4.keys())
axs[0].bar(range(len(k4)), [e4[k] for k in k4], color=C1, width=0.6)
for j, k in enumerate(k4):
    axs[0].text(j, e4[k] + 0.01, f"{e4[k]:.3f}", ha="center", fontsize=8, color=INK2)
axs[0].set_xticks(range(len(k4))); axs[0].set_xticklabels(["Aucun", "Flou\ngaussien\nσ=0,5", "Redim.\n×0,75", "Bruit\nadditif\nσ=2 HU", "Export\n8 bits"], fontsize=8)
axs[0].set_ylim(0.4, 0.85); axs[0].axhline(0.5, color=INK2, ls="--", lw=1); axs[0].set_ylabel("AUC"); axs[0].set_title("(a) Robustesse aux post-traitements"); grid(axs[0])
e3 = R["E3"]; k3 = list(e3.keys())
axs[1].bar(range(len(k3)), [e3[k][0] for k in k3], yerr=[e3[k][1] for k in k3], color=C3, width=0.6, error_kw=dict(ecolor=INK2, capsize=3))
for j, k in enumerate(k3):
    axs[1].text(j + 0.18, e3[k][0] + 0.01, f"{e3[k][0]:.2f}", fontsize=8, color=INK2)
axs[1].set_xticks(range(len(k3))); axs[1].set_xticklabels([k.capitalize() for k in k3], fontsize=8)
axs[1].set_ylim(0.3, 0.85); axs[1].axhline(0.5, color=INK2, ls="--", lw=1); axs[1].set_title("(b) Attaque non vue à l'entraînement"); grid(axs[1])
plt.tight_layout(); plt.savefig(OUT + "fig_robustesse.png", dpi=200); plt.close()

# Fig 8 : cartes de localisation
mp = np.load("maps_hybrid.npz")
fig, axs = plt.subplots(3, 4, figsize=(11, 8.4))
for k, c in enumerate((1, 2, 3, 5)):
    i = int(mp["idx"][k]); pm = mp[f"m{c}"]
    axs[0, k].imshow(win(X[i]), cmap="gray"); axs[0, k].set_title(CL[c])
    axs[1, k].imshow(win(X[i]), cmap="gray"); axs[1, k].contour(M[i], levels=[0.5], colors=[C2], linewidths=1.2)
    axs[2, k].imshow(win(X[i]), cmap="gray"); h = axs[2, k].imshow(np.ma.masked_less(pm, 0.05), cmap="magma", vmin=0, vmax=1, alpha=0.7)
    axs[2, k].contour(M[i], levels=[0.5], colors=["#1baf7a"], linewidths=1)
    for a in axs[:, k]:
        a.set_xticks([]); a.set_yticks([])
axs[0, 0].set_ylabel("Coupe falsifiée"); axs[1, 0].set_ylabel("Vérité terrain"); axs[2, 0].set_ylabel("Carte de probabilité")
cax = fig.add_axes([0.92, 0.12, 0.012, 0.22]); fig.colorbar(h, cax=cax, label="P(falsifié)")
plt.savefig(OUT + "fig_localisation.png", dpi=200, bbox_inches="tight"); plt.close()


# Fig 9 : architecture du système proposé
def box(ax, x, y, w, h, txt, fc="#eef4fc", ec=C1, fs=8.5):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=fc, ec=ec, lw=1.3))
    ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=fs, color=INK, wrap=True)


def arr(ax, a, b):
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle="-|>", mutation_scale=12, color=INK2, lw=1.2))


fig, ax = plt.subplots(figsize=(12, 5.6)); ax.set_xlim(0, 12); ax.set_ylim(0, 6); ax.axis("off")
box(ax, 0.1, 2.5, 1.6, 1.0, "PACS / modalité\nDICOM entrant", "#f4f3f0", INK2)
box(ax, 2.0, 2.4, 1.9, 1.2, "Pré-traitement\nHU, fenêtrage,\nsegmentation\npulmonaire")
box(ax, 4.2, 4.2, 2.5, 1.1, "Branche forensique\nrésidus SRM/KV, spectre,\nincohérence locale,\nduplication de blocs", "#fdf0ea", C2)
box(ax, 4.3, 2.5, 2.2, 1.1, "Branche CNN\ncouche contrainte (Bayar)\n+ EfficientNet / U-Net", "#eef4fc", C1)
box(ax, 4.3, 0.8, 2.2, 1.1, "Branche diffusion\nerreur de reconstruction\n(type DIRE / BTD)", "#eaf7f1", C3)
box(ax, 7.0, 2.5, 1.7, 1.0, "Fusion\nlocal → global\n(agrégation top-k)")
box(ax, 9.2, 3.6, 2.6, 1.0, "Décision : authentique /\nfalsifiée + type d'attaque")
box(ax, 9.2, 2.2, 2.6, 1.0, "Carte de localisation\n(probabilité par région)")
box(ax, 9.2, 0.6, 2.6, 1.1, "Module sécurité : empreinte\nSHA-256, journal d'audit,\nalerte radiologue / RSSI", "#f4f3f0", INK2)
arr(ax, (1.7, 3.0), (2.1, 3.0))
for yy in (4.75, 3.05, 1.35):
    arr(ax, (3.8, 3.0), (4.3, yy)); arr(ax, (6.5, yy), (7.0, 3.0))
arr(ax, (8.7, 3.1), (9.2, 4.1)); arr(ax, (8.7, 3.0), (9.2, 2.7)); arr(ax, (10.5, 2.2), (10.5, 1.7))
plt.savefig(OUT + "fig_architecture.png", dpi=200, bbox_inches="tight"); plt.close()

# Fig 10 : chaîne de simulation d'acquisition + menace
fig, ax = plt.subplots(figsize=(12, 2.6)); ax.set_xlim(0, 12); ax.set_ylim(0, 2.4); ax.axis("off")
steps = ["Fantôme thoracique\n(carte HU)", "Atténuation μ\n+ projection Radon", "Bruit de Poisson\n(dose I₀ variable)",
         "Reconstruction FBP\n(4 noyaux)", "Falsification\n(5 attaques)", "Jeu SimCT-Tamper\n1 080 coupes"]
for k, s in enumerate(steps):
    box(ax, 0.1 + k * 2.0, 0.7, 1.7, 1.0, s, "#fdf0ea" if k == 4 else "#eef4fc", C2 if k == 4 else C1)
    if k:
        arr(ax, (k * 2.0 - 0.2, 1.2), (0.1 + k * 2.0, 1.2))
plt.savefig(OUT + "fig_simulation.png", dpi=200, bbox_inches="tight"); plt.close()

# Fig 11 : modèle de menace
fig, ax = plt.subplots(figsize=(12, 3.4)); ax.set_xlim(0, 12); ax.set_ylim(0, 3.4); ax.axis("off")
box(ax, 0.1, 1.2, 1.8, 1.0, "Scanner CT\n(modalité)", "#f4f3f0", INK2)
box(ax, 2.6, 1.2, 1.9, 1.0, "Réseau hospitalier\n(DICOM C-STORE)", "#f4f3f0", INK2)
box(ax, 5.2, 1.2, 1.8, 1.0, "Serveur PACS", "#f4f3f0", INK2)
box(ax, 7.7, 1.2, 1.9, 1.0, "Station de lecture\nradiologue", "#f4f3f0", INK2)
box(ax, 10.2, 1.2, 1.7, 1.0, "Diagnostic /\nassurance / justice", "#f4f3f0", INK2)
for a_, b_ in [((1.9, 1.7), (2.6, 1.7)), ((4.5, 1.7), (5.2, 1.7)), ((7.0, 1.7), (7.7, 1.7)), ((9.6, 1.7), (10.2, 1.7))]:
    arr(ax, a_, b_)
box(ax, 2.7, 2.55, 4.2, 0.75, "Attaquant : interception (MitM) ou accès au PACS\n→ injection / suppression / bruit / contenu génératif", "#fdf0ea", C2)
arr(ax, (3.6, 2.55), (3.55, 2.2)); arr(ax, (6.0, 2.55), (6.1, 2.2))
box(ax, 5.0, 0.05, 4.8, 0.75, "Point de contrôle proposé : détecteur forensique + empreinte d'intégrité", "#eaf7f1", C3)
arr(ax, (6.1, 0.8), (6.1, 1.2)); arr(ax, (8.6, 0.8), (8.65, 1.2))
plt.savefig(OUT + "fig_menace.png", dpi=200, bbox_inches="tight"); plt.close()
print("ok")

In [ ]:
!python generate.py
!python experiments.py
!python hybrid.py
!python figures.py

In [ ]:
r1, r2 = json.load(open("results.json")), json.load(open("results_hybrid.json"))
print("AUC forensique globale (meilleur) :", round(r1["E1"][r1["best"]]["mean"][4], 3))
print("AUC hybride local→global        :", round(r2["hybrid"]["mean"][4], 3), "±", round(r2["hybrid"]["std"][4], 3))

## 12. Figures illustratives calculées (chapitre 2)

Ce script produit les figures calculées du mémoire : résidus SRM (figure 2.4), noyaux et bruit — NPS, loi en $1/\sqrt{I_0}$ — (figure 2.5) et détection du copier-déplacer (figure 2.6), ainsi que des illustrations complémentaires (principe CT, fenêtrage HU, manipulations classiques, FGSM, diffusion directe, traces spectrales) utiles pour la soutenance. Les figures de la revue de littérature (chapitre 1) sont, elles, reprises de publications et citées avec leur source.
Si une coupe LIDC-IDRI est disponible (cache du §3 bis), elle est sauvegardée dans `real_slice.npy` et utilisée à la place de la coupe simulée.

In [ ]:
# real_slice.npy a été enregistré au §3 bis (coupe réelle LIDC-IDRI contenant un nodule)
print("coupe réelle disponible :", Path("real_slice.npy").exists())

In [ ]:
%%writefile fig_ch1.py
"""Figures du chapitre 1 (revue de littérature), produites par calcul.

Si un fichier real_slice.npy (coupe CT réelle en HU, int16, par ex. extraite de LIDC-IDRI par le notebook)
est présent, il est utilisé ; sinon une coupe thoracique simulée (simct.py) sert d'illustration.
Usage : python fig_ch1.py [dossier_sortie]
"""
import os, sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.transform import radon, iradon
import simct as S
import features as F

OUT = sys.argv[1] if len(sys.argv) > 1 else "fig_ch1/"
os.makedirs(OUT, exist_ok=True)
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6"
INK2, GRID = "#52514e", "#e4e3df"
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": INK2, "xtick.color": INK2, "ytick.color": INK2})
rng = np.random.default_rng(7)
REAL = os.path.exists("real_slice.npy")
SRC = "coupe LIDC-IDRI" if REAL else "coupe simulée"


def win(img, L=-600, W=1500):
    return np.clip((img.astype(float) - (L - W / 2)) / W, 0, 1)


def noaxis(*axs):
    for a in axs:
        a.set_xticks([]); a.set_yticks([])


# ------------------------------------------------------------------ données de base
hu_true, lungs, nods = S.make_phantom(np.random.default_rng(11), with_nodule=True)
def lung_mask(hu):
    m = hu < -400
    lab, n = ndi.label(m)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    m = m & ~np.isin(lab, border)
    return ndi.binary_fill_holes(ndi.binary_closing(m, iterations=3))


if REAL:   # coupe réelle (LIDC-IDRI) : sert d'objet et d'image pour les figures 1.1 à 1.3, 1.7 et 1.8
    img = np.load("real_slice.npy").astype(np.int16)
    if img.shape != (S.N, S.N):
        img = np.clip(ndi.zoom(img.astype(float), S.N / img.shape[0], order=1), -1024, 3071).astype(np.int16)
    hu_true, lungs = img.astype(float), lung_mask(img)
else:
    img = S.acquire(hu_true, rng, 6e4, "ramp")

# ------------------------------------------------------------------ Fig 1 : principe CT
theta = S.THETA
mu = np.clip(S.MU_W * (1 + hu_true / 1000.0), 0, None) * S.PIX
circ = S.ellipse(hu_true.shape, S.N / 2 - 0.5, S.N / 2 - 0.5, S.N / 2 - 1, S.N / 2 - 1)
sino = radon(mu * circ, theta, circle=True)
bp = iradon(sino, theta, circle=True, filter_name=None)
fbp = iradon(sino, theta, circle=True, filter_name="ramp")
fig, ax = plt.subplots(1, 4, figsize=(13, 3.6))
ax[0].imshow(win(hu_true), cmap="gray"); ax[0].set_title("(a) Objet : carte d'atténuation")
ax[1].imshow(sino, cmap="gray", aspect="auto", extent=[0, 180, sino.shape[0], 0]); ax[1].set_title("(b) Sinogramme $p_\\theta(t)$")
ax[1].set_xlabel("angle θ (°)"); ax[1].set_ylabel("position t")
ax[2].imshow(bp, cmap="gray"); ax[2].set_title("(c) Rétroprojection non filtrée")
ax[3].imshow(win((fbp / S.PIX / S.MU_W - 1) * 1000), cmap="gray"); ax[3].set_title("(d) Rétroprojection filtrée (FBP)")
noaxis(ax[0], ax[2], ax[3]); ax[1].set_yticks([])
plt.tight_layout(); plt.savefig(OUT + "f1_principe_ct.png", dpi=200); plt.close()

# ------------------------------------------------------------------ Fig 2 : noyaux et dose -> bruit
kernels = [("rampe", "ramp"), ("Shepp-Logan", "shepp-logan"), ("cosinus", "cosine"), ("Hann", "hann")]
I0 = 4e4
counts = np.maximum(rng.poisson(I0 * np.exp(-sino)), 1)
sino_n = -np.log(counts / I0)
ys, xs = np.where(ndi.binary_erosion(lungs, iterations=6))
cy, cx = int(np.median(ys)), int(np.median(xs)) - 30
fig = plt.figure(figsize=(13, 6.2))
nps = {}
for k, (lab, ker) in enumerate(kernels):
    rec_n = (iradon(sino_n, theta, circle=True, filter_name=ker) / S.PIX / S.MU_W - 1) * 1000
    rec_0 = (iradon(sino, theta, circle=True, filter_name=ker) / S.PIX / S.MU_W - 1) * 1000
    noise = rec_n - rec_0
    a = fig.add_subplot(2, 4, k + 1)
    a.imshow(win(rec_n[cy - 40:cy + 40, cx - 40:cx + 40]), cmap="gray")
    a.set_title(f"Noyau {lab}\nσ = {noise[lungs].std():.0f} HU"); noaxis(a)
    # spectre de puissance du bruit (NPS) radial, sur des fenêtres 64x64 dans le corps
    acc = []
    for (yy, xx) in [(96, 96), (96, 160), (160, 96), (160, 160), (128, 128)]:
        p = noise[yy - 32:yy + 32, xx - 32:xx + 32]
        acc.append(np.abs(np.fft.fftshift(np.fft.fft2(p - p.mean()))) ** 2)
    P = np.mean(acc, 0); h = P.shape[0]
    yy, xx = np.mgrid[:h, :h]; r = np.sqrt((yy - h / 2) ** 2 + (xx - h / 2) ** 2).astype(int)
    nps[lab] = np.bincount(r.ravel(), P.ravel())[:33] / np.bincount(r.ravel())[:33]
a = fig.add_subplot(2, 2, 3)
for (lab, _), c in zip(kernels, (C1, C2, C3, C4)):
    f = np.arange(33) / 64 / (S.PIX / 10)  # cycles/cm
    a.plot(f, nps[lab] / nps[lab].max(), color=c, lw=2, label=lab)
a.set_xlabel("Fréquence spatiale (cycles/cm)"); a.set_ylabel("NPS normalisé"); a.set_title("(b) Spectre de puissance du bruit (NPS)")
a.legend(frameon=False, fontsize=8); a.grid(color=GRID); a.set_axisbelow(True)
a = fig.add_subplot(2, 2, 4)
doses = np.array([1e4, 2e4, 4e4, 8e4, 1.6e5, 3.2e5])
sig = []
for d0 in doses:
    cnt = np.maximum(rng.poisson(d0 * np.exp(-sino)), 1)
    rn = (iradon(-np.log(cnt / d0), theta, circle=True, filter_name="ramp") / S.PIX / S.MU_W - 1) * 1000
    r0 = (iradon(sino, theta, circle=True, filter_name="ramp") / S.PIX / S.MU_W - 1) * 1000
    sig.append((rn - r0)[lungs].std())
a.loglog(doses, sig, "o-", color=C1, lw=2, ms=6, label="mesuré (FBP rampe)")
a.loglog(doses, sig[0] * np.sqrt(doses[0] / doses), "--", color=INK2, lw=1, label="loi en $1/\\sqrt{I_0}$")
a.set_xlabel("Dose $I_0$ (photons par rayon)"); a.set_ylabel("σ du bruit (HU)"); a.set_title("(c) Bruit en fonction de la dose")
a.legend(frameon=False, fontsize=8); a.grid(color=GRID, which="both"); a.set_axisbelow(True)
fig.text(0.01, 0.97, "(a) Même acquisition reconstruite avec quatre noyaux (zoom pulmonaire)", fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.96)); plt.savefig(OUT + "f2_noyaux_bruit.png", dpi=200); plt.close()

# ------------------------------------------------------------------ Fig 3 : fenêtrage HU
fig, ax = plt.subplots(1, 4, figsize=(13, 3.5))
for a, (t, L, W) in zip(ax[:3], [("Fenêtre pulmonaire\n(L = −600, W = 1500)", -600, 1500),
                                  ("Fenêtre médiastinale\n(L = 40, W = 400)", 40, 400),
                                  ("Fenêtre osseuse\n(L = 400, W = 1800)", 400, 1800)]):
    a.imshow(win(img, L, W), cmap="gray"); a.set_title(t, fontsize=9); noaxis(a)
v = img[img > -1000].ravel()
ax[3].hist(v, bins=160, range=(-1100, 1100), color=C1)
for x0, lab in [(-1000, "air"), (-860, "poumon"), (-90, "graisse"), (45, "tissus"), (700, "os")]:
    ax[3].axvline(x0, color=INK2, lw=0.7, ls=":")
    ax[3].text(x0, 0.02, lab, rotation=90, fontsize=7, color=INK2, ha="right", va="bottom",
               transform=ax[3].get_xaxis_transform())
ax[3].set_yscale("log"); ax[3].set_xlabel("HU"); ax[3].set_title("Histogramme des HU", fontsize=9)
plt.tight_layout(); plt.savefig(OUT + "f3_fenetrage_hu.png", dpi=200); plt.close()

# ------------------------------------------------------------------ Fig 4 : manipulations classiques
base_hu, base_l, base_n = S.make_phantom(np.random.default_rng(21), with_nodule=True)
base = S.acquire(base_hu, rng, 6e4, "ramp")
for _try in range(50):
    inj = S.tamper_injection(base, base_l, np.random.default_rng(300 + _try))
    yy_, xx_ = np.where(inj[1])
    if all(np.hypot(yy_.mean() - a_, xx_.mean() - b_) > 35 for a_, b_, _ in base_n):
        break
import cv2


def clean_inpaint(im, lung, nod, rng_):
    """Suppression par inpainting de Telea limitée au disque du nodule + bruit au niveau estimé."""
    cy_, cx_, r_ = nod
    m_ = np.zeros(im.shape, np.uint8); cv2.circle(m_, (int(cx_), int(cy_)), int(r_ * 1.6 + 3), 1, -1)
    lo, hi = -1024.0, 400.0
    u8 = np.clip((im.astype(np.float32) - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)
    inp = cv2.inpaint(u8, m_, 7, cv2.INPAINT_TELEA).astype(np.float32) / 255 * (hi - lo) + lo
    out_ = im.astype(np.float32).copy()
    mb = m_.astype(bool)
    out_[mb] = inp[mb] + rng_.normal(0, S.mad_sigma(im, lung), mb.sum())
    return np.clip(np.round(out_), -1024, 3071).astype(np.int16), mb


att = [("Injection (collage)", *inj),
       ("Suppression (inpainting)", *clean_inpaint(base, base_l, base_n[0], rng)),
       ("Copier-déplacer", *S.tamper_copymove(base, base_l, rng))]
fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for i, (name, out, m) in enumerate(att):
    ys, xs = np.where(m); cy, cx = int(ys.mean()), int(xs.mean())
    y0, x0 = max(cy - 40, 0), max(cx - 40, 0)
    ax[i, 0].imshow(win(base), cmap="gray"); ax[i, 0].contour(m, [0.5], colors=[C2], linewidths=1)
    ax[i, 1].imshow(win(base[y0:y0 + 80, x0:x0 + 80]), cmap="gray")
    ax[i, 2].imshow(win(out[y0:y0 + 80, x0:x0 + 80]), cmap="gray")
    d = np.abs(out.astype(float) - base)[y0:y0 + 80, x0:x0 + 80]
    ax[i, 3].imshow(d, cmap="magma", vmin=0, vmax=np.percentile(d, 99.5) + 1)
    ax[i, 0].set_ylabel(name)
    noaxis(*ax[i])
for j, t in enumerate(["Coupe (région ciblée)", "Avant (zoom)", "Après (zoom)", "|Différence|"]):
    ax[0, j].set_title(t)
plt.tight_layout(); plt.savefig(OUT + "f4_manipulations.png", dpi=200); plt.close()

# ------------------------------------------------------------------ Fig 5 : FGSM sur un classifieur de nodules
from sklearn.linear_model import LogisticRegression
P_ = 24
pos, neg = [], []
for s in range(60):
    r_ = np.random.default_rng(100 + s)
    h_, l_, n_ = S.make_phantom(r_, with_nodule=True)
    im_ = S.acquire(h_, r_, 6e4, "ramp").astype(np.float32)
    for (cy_, cx_, rr) in n_:
        p = im_[cy_ - P_ // 2:cy_ + P_ // 2, cx_ - P_ // 2:cx_ + P_ // 2]
        if p.shape == (P_, P_): pos.append(p)
    idx = np.argwhere(ndi.binary_erosion(l_, iterations=16))
    for _ in range(3):
        cy_, cx_ = idx[r_.integers(len(idx))]
        if all(abs(cy_ - a_) > P_ or abs(cx_ - b_) > P_ for a_, b_, _ in n_):
            neg.append(im_[cy_ - P_ // 2:cy_ + P_ // 2, cx_ - P_ // 2:cx_ + P_ // 2])
X = np.array(pos + neg).reshape(len(pos) + len(neg), -1) / 1000.0
y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
clf = LogisticRegression(C=0.5, max_iter=5000).fit(X, y)
pp_ = clf.predict_proba(X[:len(pos)])[:, 1]
x0 = X[int(np.argmin(np.abs(pp_ - 0.9)))]
# gradient de la perte logistique / entrée (cible : « pas de nodule ») = (p - 0) * w  -> FGSM
p0 = clf.predict_proba(x0[None])[0, 1]
eps_list = [0, 10, 25, 50]
res = []
for eps in eps_list:
    xa = x0 + (eps / 1000.0) * np.sign(clf.coef_[0] * (p0 - 0.0)) * -1  # descente vers la classe 0
    res.append((xa, clf.predict_proba(xa[None])[0, 1]))
fig, ax = plt.subplots(1, 5, figsize=(13, 3.5))
ax[0].imshow(win(x0.reshape(P_, P_) * 1000), cmap="gray"); ax[0].set_title(f"Patch original\nP(nodule) = {p0:.2f}")
pert = (res[-1][0] - x0).reshape(P_, P_) * 1000
ax[1].imshow(pert, cmap="coolwarm", vmin=-50, vmax=50); ax[1].set_title("Perturbation ε·sign(∇)\n(ε = 50 HU)")
for k, (eps, (xa, pa)) in enumerate(zip(eps_list[1:], res[1:])):
    ax[2 + k].imshow(win(xa.reshape(P_, P_) * 1000), cmap="gray"); ax[2 + k].set_title(f"ε = {eps} HU\nP(nodule) = {pa:.2f}")
noaxis(*ax)
plt.tight_layout(); plt.savefig(OUT + "f5_fgsm.png", dpi=200); plt.close()
FGSM_INFO = dict(n_pos=len(pos), n_neg=len(neg), p0=float(p0), p=[float(r[1]) for r in res])

# ------------------------------------------------------------------ Fig 6 : processus de diffusion direct
T = 1000
betas = np.linspace(1e-4, 0.02, T); abar = np.cumprod(1 - betas)
x0d = np.clip((img.astype(float) + 1100) / 1400, 0, 1) * 2 - 1
ts = [0, 50, 150, 300, 600, 999]
fig, ax = plt.subplots(1, len(ts), figsize=(13, 3.2))
eps_ = rng.normal(size=x0d.shape)
for a, t in zip(ax, ts):
    xt = x0d if t == 0 else np.sqrt(abar[t]) * x0d + np.sqrt(1 - abar[t]) * eps_
    a.imshow(np.clip((xt + 1) / 2, 0, 1), cmap="gray"); a.set_title(f"t = {t}\n$\\sqrt{{\\bar\\alpha_t}}$ = {np.sqrt(abar[t]) if t else 1:.2f}", fontsize=9)
    noaxis(a)
plt.tight_layout(); plt.savefig(OUT + "f6_diffusion_directe.png", dpi=200); plt.close()

# ------------------------------------------------------------------ Fig 7 : traces spectrales du suréchantillonnage
f0 = img.astype(float)
low = f0[::2, ::2]
bil = ndi.zoom(low, 2, order=1)
zi = np.zeros_like(f0); zi[::2, ::2] = low
k = rng.normal(0, 1, (4, 4)); k = np.abs(k); k /= k.sum() / 4
tconv = ndi.convolve(zi, k)                                    # convolution transposée (stride 2) aléatoire


def spec(x):
    r = x - ndi.gaussian_filter(x, 1.0)
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(r))))


fig, ax = plt.subplots(1, 3, figsize=(11, 4.2))
for a, (t, x) in zip(ax, [("Coupe d'origine", f0), ("Suréchantillonnage bilinéaire", bil), ("Convolution transposée (type GAN)", tconv)]):
    a.imshow(spec(x), cmap="viridis"); a.set_title(t, fontsize=10); noaxis(a)
plt.tight_layout(); plt.savefig(OUT + "f7_traces_spectrales.png", dpi=200); plt.close()

# ------------------------------------------------------------------ Fig 8 : résidus SRM
out_i, m_i = att[0][1], att[0][2]
ys, xs = np.where(m_i); cy, cx = int(ys.mean()), int(xs.mean())
R = F.residuals(out_i)
fig, ax = plt.subplots(1, 4, figsize=(12.5, 3.4))
sl = (slice(cy - 48, cy + 48), slice(cx - 48, cx + 48))
ax[0].imshow(win(out_i[sl]), cmap="gray"); ax[0].set_title("Région injectée")
for a, (k_, t) in zip(ax[1:], [("med", "Résidu médian 3×3"), ("srm3", "Résidu SRM 3×3"), ("kv", "Résidu KV 5×5")]):
    rr = R[k_][sl]; v = np.percentile(np.abs(rr), 98)
    a.imshow(rr, cmap="gray", vmin=-v, vmax=v); a.contour(m_i[sl], [0.5], colors=[C2], linewidths=1); a.set_title(t)
noaxis(*ax)
plt.tight_layout(); plt.savefig(OUT + "f8_residus_srm.png", dpi=200); plt.close()

# ------------------------------------------------------------------ Fig 9 : copier-déplacer et appariement de blocs
out_c, m_c = att[2][1], att[2][2]
dup = F.duplicate_map(out_c, out_c > -900)
fig, ax = plt.subplots(1, 3, figsize=(11, 4.3))
ax[0].imshow(win(out_c), cmap="gray"); ax[0].set_title("Coupe falsifiée (copier-déplacer)")
ax[1].imshow(win(out_c), cmap="gray"); ax[1].contour(m_c, [0.5], colors=[C2], linewidths=1.2); ax[1].set_title("Vérité terrain (destination)")
ax[2].imshow(win(out_c), cmap="gray"); ax[2].imshow(np.ma.masked_where(~dup, dup), cmap="autumn", alpha=0.7)
ax[2].set_title("Blocs 8×8 identiques détectés\n(source et destination)")
noaxis(*ax)
plt.tight_layout(); plt.savefig(OUT + "f9_copier_deplacer.png", dpi=200); plt.close()

import json
json.dump(dict(source=SRC, fgsm=FGSM_INFO, sigma_dose=[float(s) for s in sig], doses=doses.tolist()),
          open(OUT + "fig_ch1_info.json", "w"), indent=1)
print("ok", SRC, FGSM_INFO)

In [ ]:
!python fig_ch1.py fig_ch1/
from IPython.display import Image, display
for f in sorted(glob.glob("fig_ch1/*.png")):
    display(Image(f, width=900))